# NCM-MoE Omni: Next Concept Modelling, 6 modality experts, text / image / audio

A from-scratch notebook (nothing imported from a repo). One transformer predicts **every code of a
level in a single forward**, coarse to fine: `K` forwards generate a sample instead of one per
token. Text, image and audio share one **level ladder** (level *k* has the same compression ratio
`R^(K-1-k)` in every modality), so "level 2" means the same altitude everywhere.

| piece | what it is |
|---|---|
| trunk | d=1536, L=16, GQA 16/4, QK-norm, block-causal-over-levels attention |
| experts | 3 modality groups x 2 SwiGLU experts = **6 experts, 1.66B total, ~360M active** |
| routing | group forced by the token's modality; inside the group, **expert-choice** (each expert takes its top-capacity tokens per level), so shapes are static, nothing is dropped from the graph, and a token gets 0, 1 or 2 experts (avg 1) |
| conditioning | a text prefix: text->text (prompt), text->image (caption), text->speech (transcript = TTS) |
| data | streamed straight from the Hub, **nothing cached or stored** (HF cache lives in RAM, codebooks are the only fitted state and live in the checkpoint) |
| default run | **T2 = 20B live codes** (see the tier table in the budget cell) |

Verification knobs: every knob can be overridden by an `NCM_<NAME>` environment variable, which is
how the same notebook is executed headlessly on CPU and on a small GPU (`PRESET="tiny"`).

## 0. Knobs

In [ ]:
import os, sys, time, math, json, io, gc, threading, queue, random, warnings, shutil, subprocess
T_START = time.time()
from dataclasses import dataclass

def _knob(name, default):
    """Env override `NCM_<name>` for headless runs; the literal in this cell is the default."""
    v = os.environ.get("NCM_" + name)
    if v is None:
        return default
    if isinstance(default, bool):
        return v.lower() in ("1", "true", "yes")
    return type(default)(v)

# ---- model + run size ----
PRESET   = _knob("PRESET", "full")     # "full" 1.66B total / 360M active | "tiny" same code at d=512 (any 8 GB GPU / CPU)
TIER     = _knob("TIER", "T1")         # T0 0.02B | T1 2B | T2 20B | T3 50B live codes (see budget cell)
TOKENS   = _knob("TOKENS", 0.0)        # billions of live codes; 0 = take it from TIER
STEPS    = _knob("STEPS", 0)           # >0 = hard cap on optimizer steps (smoke / CI)
SEED     = _knob("SEED", 0)

# ---- speed ----
DEVICE   = _knob("DEVICE", "auto")     # "auto" | "cuda" | "cpu"
COMPILE  = _knob("COMPILE", True)      # torch.compile the step (per pack shape, static shapes)
COMPILE_MODE = _knob("COMPILE_MODE", "default")   # "max-autotune-no-cudagraphs" for the long run
ATTN     = _knob("ATTN", "auto")       # "auto" | "flex" | "sdpa"
SWIGLU   = _knob("SWIGLU", "auto")     # "auto" | "triton" | "torch"
GPU_JPEG = _knob("GPU_JPEG", True)     # decode JPEG on the GPU (nvJPEG) instead of in the workers
MOE_CF   = _knob("MOE_CF", 0.5)        # expert capacity as a fraction of the level's tokens (2 experts x 0.5 = 1 active/token)
MICRO_TOKENS = _knob("MICRO_TOKENS", 0)   # packed positions per micro-batch; 0 = preset default
BATCH_CODES  = _knob("BATCH_CODES", 0)    # live codes per optimizer step; 0 = preset default

# ---- data (all streamed; nothing is written to disk) ----
ACT_CKPT = _knob("ACT_CKPT", -1)      # activation checkpointing: -1 = preset default, 0 = off (faster, needs ~2x activation memory; fine on 96 GB), 1 = on
STREAM_PREFETCH = _knob("STREAM_PREFETCH", 1)   # 1 = Parquet read-ahead + 128 MiB requests when the source allows it
WORKERS  = _knob("WORKERS", 0)         # processes streaming from the Hub; 0 = preset default
PREFETCH = _knob("PREFETCH", 6)        # ready micro-batches held in RAM
RAM_ONLY = _knob("RAM_ONLY", True)     # HF cache dir on /dev/shm, removed at exit
FIT_IMAGES = _knob("FIT_IMAGES", 0)    # images used to fit the frozen codebooks; 0 = preset default
FIT_AUDIO  = _knob("FIT_AUDIO", 0)
VAL_ROWS   = _knob("VAL_ROWS", 0)      # held-out rows per row type; 0 = preset default

# ---- optimisation ----
LR       = _knob("LR", 0.0)            # 0 = preset default
WARMUP   = _knob("WARMUP", 0)
WD       = _knob("WD", 0.1)
CLIP     = _knob("CLIP", 1.0)

# ---- long-run handling ----
CKPT_DIR = _knob("CKPT_DIR", "")       # e.g. /content/drive/MyDrive/ncm_moe ; "" = no checkpoints
CKPT_MIN = _knob("CKPT_MIN", 30)       # minutes between checkpoints
SESSION_HOURS = _knob("SESSION_HOURS", 11.5)   # watchdog: save and stop cleanly before the session limit
EVAL_EVERY_B  = _knob("EVAL_EVERY_B", 0.25)     # evaluate every N billion live codes
LOG_EVERY = _knob("LOG_EVERY", 10)
INSTALL  = _knob("INSTALL", True)

# ---- sources: set a name to 0 to disable it (weights inside a row type are renormalised) ----
SOURCE_WEIGHTS = {}                    # e.g. {"fineweb": 0.5, "libri": 0}; empty = defaults in the data cell

print(f"preset {PRESET} | tier {TIER} | compile {COMPILE} | attn {ATTN} | swiglu {SWIGLU} | cf {MOE_CF}")

## 0.1 Runtime: RAM-only HF cache, deps, determinism

In [ ]:
# Streaming does not write dataset shards, but the hub client keeps small metadata and lock files.
# Point them at RAM so nothing survives the process; this is asserted in the no-storage check.
_RAM_HF = None
if RAM_ONLY:
    _base = "/dev/shm" if os.path.isdir("/dev/shm") else "/tmp"
    _RAM_HF = os.path.join(_base, f"ncm_hf_{os.getpid()}")
    os.makedirs(_RAM_HF, exist_ok=True)
    for _v in ("HF_HOME", "HF_DATASETS_CACHE", "HF_HUB_CACHE", "XDG_CACHE_HOME"):
        os.environ[_v] = _RAM_HF
    import atexit
    atexit.register(lambda: shutil.rmtree(_RAM_HF, ignore_errors=True))
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
warnings.filterwarnings("ignore")

if INSTALL:
    _need = []
    for _m, _p in (("datasets", "datasets"), ("soundfile", "soundfile"), ("PIL", "pillow"), ("tqdm", "tqdm")):
        try:
            __import__(_m)
        except ImportError:
            _need.append(_p)
    if _need:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=True)

def _secret(name):
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

if _secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = _secret("HF_TOKEN")      # raises rate limits; unlocks gated sources

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| HF_TOKEN", bool(os.environ.get("HF_TOKEN")), "| HF cache ->", _RAM_HF)

## 1. Hardware and kernel probe

Every fast path is probed here and printed, so a silent fallback to a slow kernel shows up as a
line of text instead of a lower tokens/s three hours later.

In [ ]:
if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IS_CUDA = DEVICE == "cuda"
dev = torch.device(DEVICE)

# Approximate dense bf16 peaks (TFLOPS) used only to print MFU; unknown cards print TFLOPS only.
_PEAK = {"H100": 989, "A100": 312, "L4": 121, "4090": 165, "3090": 71, "3070": 40, "T4": 65, "V100": 125}
GPU_NAME, VRAM_GB, PEAK_TFLOPS = "cpu", 0.0, 0.0
if IS_CUDA:
    _p = torch.cuda.get_device_properties(0)
    GPU_NAME, VRAM_GB = _p.name, _p.total_memory / 2 ** 30
    PEAK_TFLOPS = next((v for k, v in _PEAK.items() if k in GPU_NAME), 0.0)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
BF16 = IS_CUDA and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else (torch.float16 if IS_CUDA else torch.bfloat16)

def _probe_triton():
    try:
        import triton  # noqa: F401
        return IS_CUDA
    except Exception:
        return False

def _probe_flex():
    if not IS_CUDA:
        return False
    try:
        from torch.nn.attention.flex_attention import flex_attention, create_block_mask  # noqa: F401
        return True
    except Exception:
        return False

def _probe_nvjpeg():
    if not IS_CUDA:
        return False
    try:
        import torchvision
        from torchvision.io import decode_jpeg  # noqa: F401
        return True
    except Exception:
        return False

HAS_TRITON, HAS_FLEX, HAS_NVJPEG = _probe_triton(), _probe_flex(), _probe_nvjpeg()
print(f"device {DEVICE} | {GPU_NAME} | {VRAM_GB:.0f} GiB | bf16 {BF16} | peak ~{PEAK_TFLOPS} TFLOPS")
print(f"triton {HAS_TRITON} | flex_attention {HAS_FLEX} | nvJPEG {HAS_NVJPEG} | grouped_mm {hasattr(torch, '_grouped_mm')} "
      f"| cpus {os.cpu_count()}")

## 2. Presets, the level ladder and the pack layout

**The ladder.** `K` levels, ratio `R`. Level *k* of a modality with finest length `n` has
`n / R^(K-1-k)` slots, so level 0 is the whole sample in a handful of codes and level `K-1` is the
finest detail. Images use a 2D grid (`R=4` means 2x2 pooling per level), text and audio are 1D.

**The pack.** A row is `[ text prefix | level 0 | level 1 | ... | level K-1 ]`. The prefix is the
condition (prompt / caption / transcript, bytes, always fully known). Attention is *block-causal
over levels*: a slot sees the prefix and every slot of levels <= its own, bidirectionally inside a
level. The input at level *k* is the codes of level *k-1* upsampled to length *l_k*, so a level
never sees its own targets (the leak that made an earlier version of this idea score below the
constant baseline; the correctness cell asserts it is gone).

**Codes** are 1..256 and 0 is NULL, so vocab = 257 and a real code can never be confused with padding.

In [ ]:
@dataclass
class Spec:
    K: int
    text: int            # finest text length (bytes)
    image: int           # finest image length (patches; a perfect square)
    audio: int           # finest audio length (mel frames)
    cond: int            # prefix length (bytes of prompt / caption / transcript)
    patch: int           # pixels per image patch side
    R: int = 4
    V: int = 257
    hop: int = 256
    sr: int = 16000
    n_mels: int = 80

    def lens(self, m):
        base = getattr(self, m)
        return [base // self.R ** (self.K - 1 - k) for k in range(self.K)]

    @property
    def side(self):                       # image side in pixels
        return math.isqrt(self.image) * self.patch

    @property
    def wave_len(self):                   # samples covered by `audio` frames
        return self.audio * self.hop

    def check(self):
        for m in ("text", "image", "audio"):
            assert getattr(self, m) % self.R ** (self.K - 1) == 0, f"{m} base {getattr(self, m)} not divisible by R^(K-1)"
        assert math.isqrt(self.image) ** 2 == self.image
        assert math.isqrt(self.R) ** 2 == self.R, "R must be a perfect square for 2D image pooling"
        assert all(math.isqrt(n) ** 2 == n for n in self.lens("image"))
        return self


PRESETS = {
    # same code path as "full", ~150M total / ~35M active: any 8 GB GPU, and CPU for CI
    "tiny": dict(d=512, layers=8, heads=8, kv_heads=2, ffn=1904, act_ckpt=False,
                 spec=dict(K=4, text=128, image=64, audio=512, cond=128, patch=8),
                 micro_tokens=8192, batch_codes=32768, workers=4, lr=1.5e-3, warmup=50,
                 fit_images=256, fit_audio=128, val_rows=16),
    # d=1536, L=16, expert h=3584 -> 264M per expert x 6 = 1.58B, trunk ~100M: ~1.68B total, ~360M active
    "full": dict(d=1536, layers=16, heads=16, kv_heads=4, ffn=3584, act_ckpt=True,
                 spec=dict(K=5, text=256, image=256, audio=512, cond=128, patch=16),
                 micro_tokens=32768, batch_codes=1_000_000, workers=min(16, os.cpu_count() or 8), lr=6e-4, warmup=1000,
                 fit_images=4096, fit_audio=1024, val_rows=64),
}
P = dict(PRESETS[PRESET])
SPEC = Spec(**P["spec"]).check()
N_GROUP_EXPERTS = 2
MICRO_TOKENS = MICRO_TOKENS or P["micro_tokens"]
BATCH_CODES = BATCH_CODES or P["batch_codes"]
WORKERS = WORKERS or P["workers"]
FIT_IMAGES = FIT_IMAGES or P["fit_images"]
FIT_AUDIO = FIT_AUDIO or P["fit_audio"]
VAL_ROWS = VAL_ROWS or P["val_rows"]
LR = LR or P["lr"]
WARMUP = WARMUP or P["warmup"]

ROW_TYPES = ("text0", "text", "image", "audio")      # text0 = plain text, text = prompt -> response
ROW_MOD = {"text0": "text", "text": "text", "image": "image", "audio": "audio"}
ROW_COND = {"text0": 0, "text": SPEC.cond, "image": SPEC.cond, "audio": SPEC.cond}


class Layout:
    """Static geometry of one row type: where every slot lives and which group routes it."""

    def __init__(self, spec, rtype):
        self.rtype, self.mod, self.cond = rtype, ROW_MOD[rtype], ROW_COND[rtype]
        self.K, self.lens = spec.K, spec.lens(self.mod)
        self.lvl, o = [], self.cond
        for n in self.lens:
            self.lvl.append((o, o + n)); o += n
        self.T, self.Tt = o, o - self.cond                   # packed length, target length
        bid = [0] * self.cond + [k + 1 for k, n in enumerate(self.lens) for _ in range(n)]
        self.block_id = torch.tensor(bid, dtype=torch.int32)
        self.mask = self.block_id[None, :] <= self.block_id[:, None]    # [q, kv] True = may attend
        # which MoE group routes which slots (expert-choice runs per segment = per level)
        self.segs = {}
        if self.cond:
            self.segs.setdefault("text", []).append((0, self.cond))
        self.segs.setdefault(self.mod, []).extend(self.lvl)

    def to(self, device):
        self.block_id, self.mask = self.block_id.to(device), self.mask.to(device)
        return self


LAYOUTS = {r: Layout(SPEC, r) for r in ROW_TYPES}
for _r, _l in LAYOUTS.items():
    print(f"{_r:6s} T={_l.T:5d} target={_l.Tt:5d} live codes/row={_l.Tt} levels={_l.lens}")

## 3. Kernels

| op | implementation |
|---|---|
| SwiGLU `silu(a) * b` on the fused gate+up projection | Triton: one read, one write, fused backward (registered as a `torch.library` custom op so `torch.compile` can see through it) |
| RMSNorm, residual adds, QK-norm | `F.rms_norm` + Inductor fusion |
| attention | flex-attention block mask (level-block-causal) -> SDPA with a bool mask -> CPU math |
| head matmul + cross-entropy | chunked and **checkpointed**: the `[tokens, V]` logits are never kept for backward |
| expert GEMMs | both experts of a group run as one `bmm` (capacity is uniform, so a grouped GEMM *is* a batched GEMM) |
| optimizer | `AdamW(fused=True)` |

In [ ]:
def swiglu_torch(h):
    a, b = h.chunk(2, dim=-1)
    return F.silu(a) * b

if HAS_TRITON:
    import triton
    import triton.language as tl

    @triton.jit
    def _swiglu_fwd_k(h_ptr, o_ptr, H, BLOCK: tl.constexpr):
        row = tl.program_id(0).to(tl.int64)
        cols = tl.program_id(1) * BLOCK + tl.arange(0, BLOCK)
        m = cols < H
        a = tl.load(h_ptr + row * 2 * H + cols, mask=m, other=0.0).to(tl.float32)
        b = tl.load(h_ptr + row * 2 * H + H + cols, mask=m, other=0.0).to(tl.float32)
        o = a * tl.sigmoid(a) * b
        tl.store(o_ptr + row * H + cols, o.to(o_ptr.dtype.element_ty), mask=m)

    @triton.jit
    def _swiglu_bwd_k(h_ptr, g_ptr, d_ptr, H, BLOCK: tl.constexpr):
        row = tl.program_id(0).to(tl.int64)
        cols = tl.program_id(1) * BLOCK + tl.arange(0, BLOCK)
        m = cols < H
        a = tl.load(h_ptr + row * 2 * H + cols, mask=m, other=0.0).to(tl.float32)
        b = tl.load(h_ptr + row * 2 * H + H + cols, mask=m, other=0.0).to(tl.float32)
        g = tl.load(g_ptr + row * H + cols, mask=m, other=0.0).to(tl.float32)
        s = tl.sigmoid(a)
        da = g * b * s * (1.0 + a * (1.0 - s))
        db = g * a * s
        tl.store(d_ptr + row * 2 * H + cols, da.to(d_ptr.dtype.element_ty), mask=m)
        tl.store(d_ptr + row * 2 * H + H + cols, db.to(d_ptr.dtype.element_ty), mask=m)

    @torch.library.custom_op("ncm::swiglu_fwd", mutates_args=())
    def _swiglu_fwd(h: torch.Tensor) -> torch.Tensor:
        h = h.contiguous()
        H = h.shape[-1] // 2
        o = torch.empty(*h.shape[:-1], H, device=h.device, dtype=h.dtype)
        n = h.numel() // (2 * H)
        BLOCK = min(1024, triton.next_power_of_2(H))
        _swiglu_fwd_k[(n, triton.cdiv(H, BLOCK))](h, o, H, BLOCK=BLOCK)
        return o

    @_swiglu_fwd.register_fake
    def _(h):
        return h.new_empty(*h.shape[:-1], h.shape[-1] // 2)

    @torch.library.custom_op("ncm::swiglu_bwd", mutates_args=())
    def _swiglu_bwd(h: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
        h, g = h.contiguous(), g.contiguous()
        H = h.shape[-1] // 2
        d = torch.empty_like(h)
        n = h.numel() // (2 * H)
        BLOCK = min(1024, triton.next_power_of_2(H))
        _swiglu_bwd_k[(n, triton.cdiv(H, BLOCK))](h, g, d, H, BLOCK=BLOCK)
        return d

    @_swiglu_bwd.register_fake
    def _(h, g):
        return torch.empty_like(h)

    def _swiglu_setup(ctx, inputs, output):
        ctx.save_for_backward(inputs[0])

    def _swiglu_back(ctx, g):
        (h,) = ctx.saved_tensors
        return _swiglu_bwd(h, g)

    _swiglu_fwd.register_autograd(_swiglu_back, setup_context=_swiglu_setup)
    swiglu_triton = _swiglu_fwd
else:
    swiglu_triton = None


def _pick_swiglu():
    """Parity first, then a microbenchmark: Triton wins only if it is both correct and faster."""
    if SWIGLU == "torch" or swiglu_triton is None:
        return swiglu_torch, "torch"
    h = torch.randn(4096, 2 * P["ffn"], device=dev, dtype=AMP_DTYPE, requires_grad=True)
    ref = swiglu_torch(h.float())
    got = swiglu_triton(h)
    ok = torch.allclose(got.float(), ref, atol=3e-2, rtol=3e-2)
    g = torch.randn_like(got)
    g1, = torch.autograd.grad(got, h, g)
    g2, = torch.autograd.grad(swiglu_torch(h.float()), h, g.float())
    ok = ok and torch.allclose(g1.float(), g2.float(), atol=3e-2, rtol=3e-2)
    if not ok:
        print("!! triton swiglu failed parity: using torch")
        return swiglu_torch, "torch"
    if SWIGLU == "triton":
        return swiglu_triton, "triton"
    comp = torch.compile(swiglu_torch)
    def bench(f):
        for _ in range(3):
            f(h).sum().backward()
        torch.cuda.synchronize(); t = time.time()
        for _ in range(20):
            f(h).sum().backward()
        torch.cuda.synchronize()
        return (time.time() - t) / 20
    try:
        tt, tc = bench(swiglu_triton), bench(comp)
    except Exception as e:
        print("swiglu bench skipped:", type(e).__name__)
        return swiglu_triton, "triton"
    print(f"swiglu fwd+bwd: triton {tt*1e3:.2f} ms | torch.compile {tc*1e3:.2f} ms")
    return (swiglu_triton, "triton") if tt <= tc * 1.05 else (swiglu_torch, "torch")

if IS_CUDA and HAS_TRITON:
    swiglu, SWIGLU_IMPL = _pick_swiglu()
else:
    swiglu, SWIGLU_IMPL = swiglu_torch, "torch"
print("swiglu kernel:", SWIGLU_IMPL)


# ---- chunked, checkpointed head + cross-entropy ----
def _ce_chunk(h, W, b, tgt):
    logits = F.linear(h, W, b).float()
    lse = torch.logsumexp(logits, dim=-1)
    ce = lse - logits.gather(-1, tgt[:, None]).squeeze(-1)
    return torch.stack([ce, lse])


def fused_ce(h, W, b, tgt, chunk=16384, ckpt=True):
    """h [N, d], tgt [N] -> [2, N] = (cross-entropy, logsumexp). Logits live one chunk at a time."""
    from torch.utils.checkpoint import checkpoint
    outs = []
    for i in range(0, h.shape[0], chunk):
        args = (h[i:i + chunk], W, b, tgt[i:i + chunk])
        if ckpt and torch.is_grad_enabled():
            outs.append(checkpoint(_ce_chunk, *args, use_reentrant=False))
        else:
            outs.append(_ce_chunk(*args))
    return torch.cat(outs, dim=1)

## 4. The model

* **Embeddings**: one code table per modality (vocab 257) + learned absolute position per slot
  (layouts are fixed-length and the image is 2D, so learned positions replace RoPE).
* **Attention**: GQA 16/4, QK-RMSNorm, level-block-causal mask.
* **Experts**: each modality *group* has 2 SwiGLU experts. A token only ever reaches its own
  group (forced by modality tag, so no router collapse across modalities). Inside the group routing
  is **expert-choice per level**: expert *e* takes the `ceil(cf * l)` tokens of the level it likes
  best. Capacity is therefore exact, shapes are static, there is no load-balance loss, and a token
  gets 0, 1 or 2 experts (1 on average at `cf=0.5`). Choosing among the tokens of **one level**
  (never across levels) keeps routing from leaking information from finer levels.

In [ ]:
from torch.utils.checkpoint import checkpoint as _ckpt


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return F.rms_norm(x, (x.shape[-1],), self.w.to(x.dtype), self.eps)


_flex_fn = None
# Smaller tiles: the default 128x128 forward tile at head_dim 96 needs ~178 KB of shared memory, over the A100's 163 KB limit.
FLEX_OPTS = dict(BLOCK_M=64, BLOCK_N=64, BLOCK_M1=32, BLOCK_N1=64, BLOCK_M2=64, BLOCK_N2=32)
def _flex():
    global _flex_fn
    if _flex_fn is None:
        from torch.nn.attention.flex_attention import flex_attention
        _flex_fn = flex_attention if torch.compiler.is_compiling() else torch.compile(flex_attention)
    return _flex_fn


class Attn(nn.Module):
    def __init__(self, d, nh, nkv):
        super().__init__()
        self.nh, self.nkv, self.hd = nh, nkv, d // nh
        self.qkv = nn.Linear(d, (nh + 2 * nkv) * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.qn, self.kn = RMSNorm(self.hd), RMSNorm(self.hd)

    def forward(self, x, lay, use_flex):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split([self.nh * self.hd, self.nkv * self.hd, self.nkv * self.hd], dim=-1)
        q = self.qn(q.view(B, T, self.nh, self.hd)).transpose(1, 2)
        k = self.kn(k.view(B, T, self.nkv, self.hd)).transpose(1, 2)
        v = v.view(B, T, self.nkv, self.hd).transpose(1, 2)
        if use_flex:
            y = _flex()(q, k, v, block_mask=lay.flex_mask, enable_gqa=True, kernel_options=FLEX_OPTS)
        else:
            r = self.nh // self.nkv
            y = F.scaled_dot_product_attention(q, k.repeat_interleave(r, 1), v.repeat_interleave(r, 1),
                                               attn_mask=lay.mask)
        return self.o(y.transpose(1, 2).reshape(B, T, -1))


class GroupExperts(nn.Module):
    """E SwiGLU experts for one modality; expert-choice routing per segment (= per level)."""

    def __init__(self, d, h, E, cf):
        super().__init__()
        self.E, self.h, self.cf = E, h, cf
        self.router = nn.Linear(d, E, bias=False)
        self.w1 = nn.Parameter(torch.empty(E, d, 2 * h))      # gate | up, one GEMM
        self.w2 = nn.Parameter(torch.empty(E, h, d))
        self.record = False
        self.last_cov = None

    def forward(self, x, segs):
        B, T, d = x.shape
        E = self.E
        idxs, ws = [], []
        for a, b in segs:
            l = b - a
            c = max(1, math.ceil(self.cf * l))
            p = self.router(x[:, a:b]).float().softmax(-1)             # [B, l, E]
            w, i = p.transpose(1, 2).topk(c, dim=-1)                   # each expert's top-c tokens
            idxs.append(i + a); ws.append(w)
        idx, w = torch.cat(idxs, -1), torch.cat(ws, -1)               # [B, E, C]
        C = idx.shape[-1]
        xg = torch.gather(x.unsqueeze(1).expand(B, E, T, d), 2, idx.unsqueeze(-1).expand(B, E, C, d))
        xg = xg.permute(1, 0, 2, 3).reshape(E, B * C, d)
        hh = swiglu(torch.bmm(xg, self.w1.to(xg.dtype)))              # [E, BC, h]
        y = torch.bmm(hh, self.w2.to(xg.dtype)).view(E, B, C, d).permute(1, 0, 2, 3)
        y = y * w.unsqueeze(-1).to(y.dtype)
        out = x.new_zeros(B, T, d)
        out.scatter_add_(1, idx.reshape(B, E * C, 1).expand(B, E * C, d), y.reshape(B, E * C, d).to(out.dtype))
        if self.record:
            with torch.no_grad():
                cnt = torch.zeros(B, T, device=x.device).scatter_add_(1, idx.reshape(B, E * C), torch.ones(B, E * C, device=x.device))
                live = torch.zeros(T, dtype=torch.bool, device=x.device)
                for a, b in segs:
                    live[a:b] = True
                cl = cnt[:, live]
                self.last_cov = torch.stack([(cl == j).float().mean() for j in range(E + 1)])
        return out


class Block(nn.Module):
    def __init__(self, d, nh, nkv, h, E, cf):
        super().__init__()
        self.ln1, self.ln2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attn(d, nh, nkv)
        self.experts = nn.ModuleDict({m: GroupExperts(d, h, E, cf) for m in ("text", "image", "audio")})

    def forward(self, x, lay, use_flex):
        x = x + self.attn(self.ln1(x), lay, use_flex)
        y = self.ln2(x)
        moe = None
        for g, segs in lay.segs.items():                 # only the groups this row type touches
            o = self.experts[g](y, segs)
            moe = o if moe is None else moe + o
        return x + moe


def up_codes(codes, m, spec):
    """Level k-1 codes -> level k grid (nearest). Image is 2D (2x2), text/audio are 1D."""
    r = spec.R
    if m == "image":
        B, n = codes.shape
        g, f = math.isqrt(n), math.isqrt(r)
        return codes.view(B, g, g).repeat_interleave(f, 1).repeat_interleave(f, 2).reshape(B, -1)
    return codes.repeat_interleave(r, 1)


class Model(nn.Module):
    def __init__(self, spec, d, layers, heads, kv_heads, ffn, cf, act_ckpt, E=2):
        super().__init__()
        self.spec, self.d, self.act_ckpt = spec, d, act_ckpt
        mods = ("text", "image", "audio")
        self.emb = nn.ModuleDict({m: nn.Embedding(spec.V, d) for m in mods})
        self.pos = nn.ParameterDict({m: nn.Parameter(torch.zeros(sum(spec.lens(m)), d)) for m in mods})
        self.pos_cond = nn.Parameter(torch.zeros(max(spec.cond, 1), d))
        self.blocks = nn.ModuleList([Block(d, heads, kv_heads, ffn, E, cf) for _ in range(layers)])
        self.norm = RMSNorm(d)
        self.heads = nn.ModuleDict({m: nn.Linear(d, spec.V) for m in mods})
        self.use_flex = False
        self.lay = None
        self.apply(self._init)
        resid = 0.02 / math.sqrt(2 * layers)
        for blk in self.blocks:
            nn.init.normal_(blk.attn.o.weight, std=resid)
            for g in blk.experts.values():
                nn.init.normal_(g.w1, std=0.02)
                nn.init.normal_(g.w2, std=resid)
                nn.init.normal_(g.router.weight, std=0.02)
        for p_ in list(self.pos.values()) + [self.pos_cond]:
            nn.init.normal_(p_, std=0.02)
        for hd in self.heads.values():                  # zero head: step-0 prediction is uniform, loss = ln(257) at any width
            nn.init.zeros_(hd.weight)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)

    def hidden(self, rtype, cond, codes):
        """cond [B, Lc] long | None, codes: K tensors [B, l_k] long -> [B, T, d].
        Level k's input is the upsampled codes of level k-1; level 0 reads the NULL code."""
        lay, m, spec = self.lay[rtype], ROW_MOD[rtype], self.spec
        B = codes[0].shape[0]
        parts = []
        if lay.cond:
            parts.append(self.emb["text"](cond) + self.pos_cond[:lay.cond])
        E, o = self.emb[m], 0
        for k, n in enumerate(lay.lens):
            ids = codes[0].new_zeros(B, n) if k == 0 else up_codes(codes[k - 1], m, spec)
            parts.append(E(ids) + self.pos[m][o:o + n])
            o += n
        x = torch.cat(parts, 1)
        for blk in self.blocks:
            if self.act_ckpt and self.training and torch.is_grad_enabled():
                x = _ckpt(blk, x, lay, self.use_flex, use_reentrant=False)
            else:
                x = blk(x, lay, self.use_flex)
        return self.norm(x)

    def loss(self, rtype, cond, codes):
        lay, m = self.lay[rtype], ROW_MOD[rtype]
        h = self.hidden(rtype, cond, codes)[:, lay.cond:]
        B, Tt, d = h.shape
        tgt = torch.cat(codes, 1)
        head = self.heads[m]
        out = fused_ce(h.reshape(-1, d), head.weight, head.bias, tgt.reshape(-1))
        ce, lse = out[0].view(B, Tt), out[1].view(B, Tt)
        loss = (ce + 1e-4 * lse * lse).mean()                      # light z-loss keeps logits bounded
        per_level = torch.stack([ce[:, a - lay.cond:b - lay.cond].mean() for a, b in lay.lvl]).detach()
        return loss, per_level

    def level_logits(self, rtype, h, k):
        lay, m = self.lay[rtype], ROW_MOD[rtype]
        a, b = lay.lvl[k]
        return self.heads[m](h[:, a:b]).float()


def build_model(meta=False):
    ctx = torch.device("meta") if meta else torch.device("cpu")
    with ctx:
        net = Model(SPEC, P["d"], P["layers"], P["heads"], P["kv_heads"], P["ffn"], MOE_CF, P["act_ckpt"] if ACT_CKPT < 0 else bool(ACT_CKPT))
    return net


def param_report(net):
    """total, trunk, per-expert, and active-per-token for each modality."""
    tot = sum(p.numel() for p in net.parameters())
    ex = {g: 0 for g in ("text", "image", "audio")}
    for blk in net.blocks:
        for g, mod in blk.experts.items():
            ex[g] += sum(p.numel() for n, p in mod.named_parameters() if n != "router.weight")
    experts_total = sum(ex.values())
    E = net.blocks[0].experts["text"].E
    per_expert = ex["text"] / E
    trunk = tot - experts_total
    act = {}
    for m in ex:
        own_emb = net.emb[m].weight.numel() + net.pos[m].numel() + sum(p.numel() for p in net.heads[m].parameters())
        other = sum(net.emb[o].weight.numel() + net.pos[o].numel() + sum(p.numel() for p in net.heads[o].parameters())
                    for o in ex if o != m)
        act[m] = trunk - other + ex[m] * (MOE_CF * E) / E
    return dict(total=tot, trunk=trunk, experts_total=experts_total, per_expert=per_expert,
                active=act, active_max=max(act.values()))

## 5. Frozen codecs (all on the GPU)

No learned encoder: every modality is turned into the **same ladder** of 256-way codes by a fixed
procedure, so "level k" is comparable across text, image and audio.

| modality | finest level (K-1) | coarser levels | decode |
|---|---|---|---|
| text | the byte itself (exact) | span-pooled random byte projection, sign bits = 8-bit locality-sensitive code | bytes -> UTF-8 |
| image | VQ of a `patch x patch x 3` pixel block | 2x2 average-pooled grid (a blurry image), VQ per level | codeword tiles |
| audio | VQ of an 80-bin log-mel frame (16 ms hop) | rate pyramid (average-pool in time), VQ per level | log-mel -> pseudo-inverse -> Griffin-Lim |

Codebooks are **fit once at startup** by k-means on a streamed sample that is held in RAM and then
discarded; they are tiny and live in the checkpoint, so a resumed run uses the same codes.
Quality of image/audio generations is bounded by these codecs: they are coarse by construction.

In [ ]:
class TextCodec:
    def __init__(self, spec, device, seed=1234):
        self.spec, self.device = spec, device
        g = torch.Generator().manual_seed(seed)
        self.planes = torch.randn(256, 8, generator=g).to(device)              # byte -> 8 hash planes
        self.pw = (1 << torch.arange(8, device=device)).long()

    @torch.no_grad()
    def encode(self, b):
        """b [B, L] uint8 -> K long tensors [B, l_k] with codes in 1..256."""
        K, R = self.spec.K, self.spec.R
        b = b.long()
        codes = [None] * K
        codes[K - 1] = b + 1
        proj = self.planes[b].transpose(1, 2)                                   # [B, 8, L]
        for j in range(K - 1):
            pooled = F.avg_pool1d(proj, R ** (K - 1 - j)).transpose(1, 2)       # [B, l_j, 8]
            codes[j] = 1 + ((pooled > 0).long() * self.pw).sum(-1)
        return codes

    @staticmethod
    def decode(finest):
        return [bytes((c - 1).clamp(0, 255).to(torch.uint8).cpu().tolist()).rstrip(b"\x00").decode("utf-8", "replace")
                for c in finest]


def kmeans(x, k=256, iters=25, seed=0):
    """x [N, D] float -> centroids [k, D]. Lloyd on the GPU with dead-centroid re-seeding."""
    g = torch.Generator(device=x.device).manual_seed(seed)
    if x.shape[0] > 200_000:
        x = x[torch.randperm(x.shape[0], generator=g, device=x.device)[:200_000]]
    N = x.shape[0]
    if N < k:
        x = torch.cat([x, x[torch.randint(N, (k - N,), generator=g, device=x.device)]
                       + 1e-3 * torch.randn(k - N, x.shape[1], device=x.device, generator=g)])
        N = k
    cent = x[torch.randperm(N, generator=g, device=x.device)[:k]].clone()
    x2 = (x * x).sum(1, keepdim=True)
    for _ in range(iters):
        a = (x2 - 2 * x @ cent.T + (cent * cent).sum(1)).argmin(1)
        sums = torch.zeros_like(cent).index_add_(0, a, x)
        cnt = torch.bincount(a, minlength=k)
        new = sums / cnt.clamp(min=1)[:, None]
        dead = cnt == 0
        if dead.any():
            new[dead] = x[torch.randint(N, (int(dead.sum()),), generator=g, device=x.device)]
        cent = new
    return cent


def sample_points(feats, cap):
    """Keep at most `cap` random rows of each level's features, so fitting never holds a whole corpus."""
    out = []
    for f in feats:
        f = f.reshape(-1, f.shape[-1])
        out.append(f if f.shape[0] <= cap else f[torch.randperm(f.shape[0], device=f.device)[:cap]])
    return out


def vq(x, cb):
    """x [..., D], cb [k, D] -> 1-based codes [...]. argmin ||x-c||^2 via one matmul."""
    sh = x.shape[:-1]
    xf = x.reshape(-1, x.shape[-1])
    d = (cb * cb).sum(1)[None, :] - 2 * xf @ cb.T
    return (d.argmin(1) + 1).view(sh)


class ImageCodec:
    def __init__(self, spec, device):
        self.spec, self.device = spec, device
        self.G, self.p = math.isqrt(spec.image), spec.patch
        self.D = 3 * self.p * self.p
        self.cb = None                                                           # list K x [256, D]

    def features(self, img):
        """img uint8 [B, 3, S, S] -> list K of float [B, n_k, D]; level j is the grid pooled R^(K-1-j)x."""
        K, f = self.spec.K, math.isqrt(self.spec.R)
        x = img.float() / 255.0 - 0.5
        B = x.shape[0]
        cur = F.unfold(x, self.p, stride=self.p).view(B, self.D, self.G, self.G)
        feats = [None] * K
        for j in reversed(range(K)):
            feats[j] = cur.flatten(2).transpose(1, 2).contiguous()
            if j > 0:
                cur = F.avg_pool2d(cur, f)
        return feats

    @torch.no_grad()
    def fit(self, feats_list):
        """feats_list: per batch, a list over levels of [m, D] points (see `sample_points`); one codebook per level."""
        self.cb = [kmeans(torch.cat([f[j].reshape(-1, self.D) for f in feats_list]), 256, seed=j)
                   for j in range(self.spec.K)]

    @torch.no_grad()
    def encode(self, img):
        return [vq(f, self.cb[j]) for j, f in enumerate(self.features(img))]

    @torch.no_grad()
    def decode(self, codes, level=None):
        """codes [B, n_j] at `level` (default finest) -> uint8 [B, 3, S, S]."""
        level = self.spec.K - 1 if level is None else level
        cells = self.cb[level][(codes - 1).clamp(min=0)]                         # [B, n, D]
        g = math.isqrt(codes.shape[1])
        img = F.fold(cells.transpose(1, 2), (g * self.p, g * self.p), self.p, stride=self.p)
        S = self.spec.side
        if img.shape[-1] != S:
            img = F.interpolate(img, size=(S, S), mode="nearest")
        return ((img + 0.5).clamp(0, 1) * 255).round().to(torch.uint8)

    def state(self):
        return None if self.cb is None else [c.cpu() for c in self.cb]

    def load(self, st):
        self.cb = [c.to(self.device) for c in st]


def decode_jpeg_gpu(blobs, side, device):
    """list of JPEG byte strings -> uint8 [B, 3, side, side] (short side resized, centre-cropped), nvJPEG if available."""
    from torchvision.io import decode_jpeg, ImageReadMode
    ts = [torch.frombuffer(bytearray(b), dtype=torch.uint8) for b in blobs]
    try:
        imgs = decode_jpeg(ts, device=device, mode=ImageReadMode.RGB)
    except Exception:
        imgs = [decode_jpeg(t, mode=ImageReadMode.RGB).to(device) for t in ts]
    out = []
    for im in imgs:
        _, H, W = im.shape
        s = side / min(H, W)
        nh, nw = max(side, round(H * s)), max(side, round(W * s))
        im = F.interpolate(im[None].float(), size=(nh, nw), mode="bilinear", antialias=True, align_corners=False)[0]
        t, l = (nh - side) // 2, (nw - side) // 2
        out.append(im[:, t:t + side, l:l + side].round().clamp(0, 255).to(torch.uint8))
    return torch.stack(out)


def _mel_fb(n_freqs, n_mels, sr):
    """Slaney-style mel filterbank [n_freqs, n_mels] (no torchaudio dependency)."""
    def hz2mel(f):
        f = torch.as_tensor(f, dtype=torch.float64)
        lin = f / (200.0 / 3)
        log = 15.0 + torch.log(f.clamp(min=1000.0) / 1000.0) / (math.log(6.4) / 27.0)
        return torch.where(f >= 1000.0, log, lin)
    def mel2hz(m):
        m = torch.as_tensor(m, dtype=torch.float64)
        lin = (200.0 / 3) * m
        log = 1000.0 * torch.exp((math.log(6.4) / 27.0) * (m - 15.0))
        return torch.where(m >= 15.0, log, lin)
    all_f = torch.linspace(0, sr / 2, n_freqs, dtype=torch.float64)
    pts = mel2hz(torch.linspace(float(hz2mel(0.0)), float(hz2mel(sr / 2)), n_mels + 2, dtype=torch.float64))
    diff = pts[1:] - pts[:-1]
    slopes = pts[None, :] - all_f[:, None]
    down = -slopes[:, :-2] / diff[:-1]
    up = slopes[:, 2:] / diff[1:]
    fb = torch.clamp(torch.minimum(down, up), min=0.0)
    fb = fb * (2.0 / (pts[2:n_mels + 2] - pts[:n_mels]))[None, :]
    return fb.float()


class AudioCodec:
    N_FFT = 1024

    def __init__(self, spec, device):
        self.spec, self.device = spec, device
        self.fb = _mel_fb(self.N_FFT // 2 + 1, spec.n_mels, spec.sr).to(device)    # [513, 80]
        self.pinv = torch.linalg.pinv(self.fb.cpu()).to(device)                    # [80, 513]
        self.win = torch.hann_window(self.N_FFT, device=device)
        self.mean, self.std = -5.0, 2.5
        self.cb = None

    def logmel(self, wave):
        """wave float [B, n] -> log-mel [B, audio, 80] (n = audio * hop)."""
        s = torch.stft(wave, self.N_FFT, self.spec.hop, window=self.win, center=True, return_complex=True)
        mel = (s.abs().pow(2).transpose(1, 2) @ self.fb)[:, :self.spec.audio]
        return torch.log(mel.clamp(min=1e-5))

    def features(self, wave):
        K, R = self.spec.K, self.spec.R
        x = ((self.logmel(wave) - self.mean) / self.std).transpose(1, 2)          # [B, 80, F]
        return [F.avg_pool1d(x, R ** (K - 1 - j)).transpose(1, 2).contiguous() if j < K - 1
                else x.transpose(1, 2).contiguous() for j in range(K)]

    @torch.no_grad()
    def set_stats(self, waves):
        lm = torch.cat([self.logmel(w) for w in waves]).flatten()
        self.mean, self.std = float(lm.mean()), float(lm.std().clamp(min=1e-3))

    @torch.no_grad()
    def fit(self, feats_list):
        self.cb = [kmeans(torch.cat([f[j].reshape(-1, self.spec.n_mels) for f in feats_list]), 256, seed=j)
                   for j in range(self.spec.K)]

    @torch.no_grad()
    def encode(self, wave):
        return [vq(f, self.cb[j]) for j, f in enumerate(self.features(wave))]

    @torch.no_grad()
    def decode(self, codes, level=None, gl_iters=64):
        """codes [B, F] -> waveform float [B, n]: codeword log-mel -> power spec (pinv) -> Griffin-Lim."""
        level = self.spec.K - 1 if level is None else level
        z = self.cb[level][(codes - 1).clamp(min=0)]                              # [B, n, 80]
        if z.shape[1] != self.spec.audio:
            z = z.repeat_interleave(self.spec.audio // z.shape[1], 1)
        mel = torch.exp(z * self.std + self.mean)
        mag = (mel @ self.pinv).clamp(min=1e-8).sqrt().transpose(1, 2)            # [B, 513, F]
        return griffin_lim(mag, self.win, self.spec.hop, self.N_FFT, gl_iters)

    def state(self):
        return None if self.cb is None else dict(cb=[c.cpu() for c in self.cb], mean=self.mean, std=self.std)

    def load(self, st):
        self.cb, self.mean, self.std = [c.to(self.device) for c in st["cb"]], st["mean"], st["std"]


@torch.no_grad()
def griffin_lim(mag, win, hop, n_fft, iters=64, momentum=0.99):
    """Fast Griffin-Lim. mag [B, n_freq, F] -> waveform [B, (F-1)*hop]."""
    g = torch.Generator(device=mag.device).manual_seed(0)
    phase = torch.exp(2j * math.pi * torch.rand(mag.shape, device=mag.device, generator=g))
    spec, prev = mag * phase, None
    for _ in range(iters):
        wav = torch.istft(spec, n_fft, hop, window=win, center=True)
        est = torch.stft(wav, n_fft, hop, window=win, center=True, return_complex=True)[..., :mag.shape[-1]]
        if prev is not None:
            est = est + momentum * (est - prev)
        prev = est
        spec = mag * est / est.abs().clamp(min=1e-8)
    return torch.istft(spec, n_fft, hop, window=win, center=True)


class Codecs:
    def __init__(self, spec, device):
        self.text, self.image, self.audio = TextCodec(spec, device), ImageCodec(spec, device), AudioCodec(spec, device)

    def ready(self):
        return self.image.cb is not None and self.audio.cb is not None

    def state(self):
        return dict(image=self.image.state(), audio=self.audio.state())

    def load(self, st):
        self.image.load(st["image"]); self.audio.load(st["audio"])

## 6. Data: streamed from the Hub, nothing stored

```
Hub (streaming=True) -> W worker processes (decode, cheap) -> micro-batch builder
     -> GPU codecs on a side CUDA stream -> RAM ring (PREFETCH batches) -> trainer
```

* **No shards, no npz, no dataset cache, no codec dumps.** The only buffer is the in-RAM ring.
* Rows are **modality-homogeneous** (text-only, prompt->response, caption->image, transcript->speech),
  so there are exactly 4 static pack shapes and no wasted NULL slots.
* The stream is split 3 ways by shard (or by row index when a dataset has few shards):
  part 0 = validation, part 1 = codebook fit, parts 2.. = training workers. Splits never overlap.
* A source that errors is retried with backoff and dropped after repeated failures (gated or renamed
  datasets cost their own weight, not the run).

Sources are what open pretraining recipes use: FineWeb-Edu / Wikipedia / Cosmopedia / code / OpenWebMath
(text), NuminaMath + SmolTalk (instruction), CC12M / CC3M / COCO / DocVQA (image+caption),
LibriSpeech / LibriTTS-R / People's Speech (audio+transcript).

In [ ]:
import re as _re


@dataclass
class Source:
    name: str
    kind: str                      # text0 | text | image | audio
    path: str
    config: str = None
    split: str = "train"
    weight: float = 1.0
    raw: dict = None               # column -> "image" | "audio" (kept as undecoded bytes)
    buf: int = 512                 # shuffle buffer (rows, in RAM)
    fn: str = ""                   # extractor name


SOURCES = [
    # --- text, plain pretraining (text0)
    Source("fineweb", "text0", "HuggingFaceFW/fineweb-edu", "sample-10BT", weight=0.45, fn="t_text"),
    Source("wiki", "text0", "wikimedia/wikipedia", "20231101.en", weight=0.15, fn="t_text"),
    Source("cosmo", "text0", "HuggingFaceTB/smollm-corpus", "cosmopedia-v2", weight=0.15, fn="t_text"),
    Source("code", "text0", "codeparrot/codeparrot-clean", weight=0.10, fn="t_content"),
    Source("owm", "text0", "open-web-math/open-web-math", weight=0.15, fn="t_text"),
    # --- text, prompt -> response
    Source("numina", "text", "AI-MO/NuminaMath-CoT", weight=0.5, fn="p_numina"),
    Source("smoltalk", "text", "HuggingFaceTB/smoltalk", "all", weight=0.5, fn="p_smoltalk"),
    # --- image + caption
    Source("cc12m", "image", "pixparse/cc12m-wds", weight=0.55, raw={"jpg": "image"}, buf=128, fn="i_wds"),
    Source("cc3m", "image", "pixparse/cc3m-wds", weight=0.25, raw={"jpg": "image"}, buf=128, fn="i_wds"),
    Source("docvqa", "image", "HuggingFaceM4/DocumentVQA", weight=0.15, raw={"image": "image"}, buf=64, fn="i_docvqa"),
    Source("coco", "image", "lmms-lab/COCO-Caption2017", split="val", weight=0.05, raw={"image": "image"}, buf=64, fn="i_coco"),
    # --- audio + transcript
    Source("libri", "audio", "openslr/librispeech_asr", "clean", "train.100", weight=0.40, raw={"audio": "audio"}, buf=64, fn="a_text"),
    Source("libritts", "audio", "mythicinfinity/libritts_r", "clean", "train.clean.100", weight=0.30, raw={"audio": "audio"}, buf=64, fn="a_tts"),
    Source("peoples", "audio", "MLCommons/peoples_speech", "clean", weight=0.30, raw={"audio": "audio"}, buf=64, fn="a_text"),
]
for _s in SOURCES:
    if _s.name in SOURCE_WEIGHTS:
        _s.weight = float(SOURCE_WEIGHTS[_s.name])
SOURCES = [s for s in SOURCES if s.weight > 0]


def _bytes(s, n):
    b = (s or "").encode("utf-8", "ignore")[:n]
    a = np.zeros(n, np.uint8)
    a[:len(b)] = np.frombuffer(b, np.uint8)
    return a, len(b)


def _clean(s):
    return _re.sub(r"\s+", " ", s or "").strip()


# ---- extractors: HF example -> list of row dicts (numpy / bytes only, so they pickle cheaply) ----
def _x_t_text(ex, spec, rng, key="text"):
    raw = (ex.get(key) or "").encode("utf-8", "ignore")
    L = spec.text
    if len(raw) < L // 2:
        return []
    out = []
    n = min(4, max(1, len(raw) // L))
    for _ in range(n):
        st = rng.randint(0, max(0, len(raw) - L))
        a = np.zeros(L, np.uint8); c = raw[st:st + L]; a[:len(c)] = np.frombuffer(c, np.uint8)
        out.append(dict(t="text0", tgt=a))
    return out

def x_t_text(ex, spec, rng): return _x_t_text(ex, spec, rng, "text")
def x_t_content(ex, spec, rng): return _x_t_text(ex, spec, rng, "content")


def _prompt_row(p, r, spec):
    p, r = (p or "").strip(), (r or "").strip()
    if len(p) < 4 or len(r) < 8:
        return []
    return [dict(t="text", cond=_bytes(p, spec.cond)[0], tgt=_bytes(r, spec.text)[0])]

def x_p_numina(ex, spec, rng): return _prompt_row(ex.get("problem"), ex.get("solution"), spec)

def x_p_smoltalk(ex, spec, rng):
    m = ex.get("messages") or []
    u = next((x["content"] for x in m if x.get("role") == "user"), None)
    a = next((x["content"] for x in m if x.get("role") == "assistant"), None)
    return _prompt_row(u, a, spec)


def _img_row(img, cap, spec):
    """img: {'bytes':..} from an undecoded Image column. JPEGs stay as bytes for nvJPEG; others are decoded here."""
    cap = _clean(cap)
    if not cap or not img or not img.get("bytes"):
        return []
    raw = img["bytes"]
    row = dict(t="image", cond=_bytes(cap, spec.cond)[0])
    if GPU_JPEG and HAS_NVJPEG and raw[:2] == b"\xff\xd8":
        row["jpeg"] = raw
        return [row]
    from PIL import Image
    im = Image.open(io.BytesIO(raw))
    S = spec.side
    im.draft("RGB", (S * 2, S * 2))
    im = im.convert("RGB")
    w, h = im.size
    s = S / min(w, h)
    im = im.resize((max(S, round(w * s)), max(S, round(h * s))), Image.BICUBIC)
    w, h = im.size
    l, t = (w - S) // 2, (h - S) // 2
    row["img"] = np.asarray(im.crop((l, t, l + S, t + S)), dtype=np.uint8).transpose(2, 0, 1).copy()
    return [row]

def x_i_wds(ex, spec, rng): return _img_row(ex.get("jpg"), ex.get("txt"), spec)
def x_i_coco(ex, spec, rng):
    a = ex.get("answer")
    return _img_row(ex.get("image"), a[0] if isinstance(a, list) and a else a, spec)
def x_i_docvqa(ex, spec, rng):
    a = ex.get("answers") or [""]
    return _img_row(ex.get("image"), f"{ex.get('question', '')} {a[0] if a else ''}", spec)


def _wave(aud, spec):
    import soundfile as sf
    data, sr = sf.read(io.BytesIO(aud["bytes"]), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(1)
    if sr != spec.sr:
        x = torch.from_numpy(data)[None, None]
        data = F.interpolate(x, size=int(len(data) * spec.sr / sr), mode="linear", align_corners=False)[0, 0].numpy()
    return data


def _aud_row(aud, text, spec):
    text = _clean(text).lower()
    if not text or not aud or not aud.get("bytes"):
        return []
    w = _wave(aud, spec)
    n = spec.wave_len
    if len(w) > n or len(w) < spec.sr // 2:           # a crop would no longer match the transcript
        return []
    pcm = np.zeros(n, np.int16)
    pcm[:len(w)] = np.clip(w * 32767, -32768, 32767).astype(np.int16)
    return [dict(t="audio", cond=_bytes(text, spec.cond)[0], wav=pcm)]

def x_a_text(ex, spec, rng): return _aud_row(ex.get("audio"), ex.get("text"), spec)
def x_a_tts(ex, spec, rng): return _aud_row(ex.get("audio"), ex.get("text_normalized") or ex.get("text_original"), spec)

EXTRACT = {k[2:]: v for k, v in list(globals().items()) if k.startswith("x_")}


def open_stream(src, part, nparts, seed):
    from datasets import load_dataset, Audio, Image
    kw = dict(split=src.split, streaming=True)
    ds = None
    if STREAM_PREFETCH:                              # Parquet read-ahead + 128 MiB requests (HF "streaming datasets" blog)
        try:
            import pyarrow, pyarrow.dataset as pds
            fso = pds.ParquetFragmentScanOptions(cache_options=pyarrow.CacheOptions(prefetch_limit=1, range_size_limit=128 << 20))
            ds = (load_dataset(src.path, src.config, fragment_scan_options=fso, **kw) if src.config
                  else load_dataset(src.path, fragment_scan_options=fso, **kw))
        except Exception:
            ds = None                                # non-Parquet builder or older `datasets`: plain stream
    if ds is None:
        ds = load_dataset(src.path, src.config, **kw) if src.config else load_dataset(src.path, **kw)
    if part == 1 and nparts > 1:
        print(f"    [{src.name}] shards={ds.n_shards} (streams {nparts - 1})", flush=True)
    for col, kind in (src.raw or {}).items():
        ds = ds.cast_column(col, Audio(decode=False) if kind == "audio" else Image(decode=False))
    if nparts > 1:
        if (ds.n_shards or 1) >= nparts:
            ds = ds.shard(num_shards=nparts, index=part)
        else:                                        # few shards: partition by row index (reads more, wastes nothing else)
            ds = ds.filter(lambda ex, i, _p=part, _n=nparts: i % _n == _p, with_indices=True)
    return iter(ds.shuffle(seed=seed, buffer_size=src.buf))


def rows_of(src, spec, part, nparts, seed, rng, log=None):
    """Endless generator of rows from one source; reconnects with backoff, gives up after repeated failure."""
    fn, fails, epoch = EXTRACT[src.fn], 0, 0
    while True:
        try:
            for ex in open_stream(src, part, nparts, seed + 977 * epoch):
                for r in fn(ex, spec, rng):
                    r["src"] = src.name
                    fails = 0
                    yield r
            epoch += 1                                # stream exhausted: reshuffle and go again
        except Exception as e:
            fails += 1
            if log:
                log(f"[{src.name}] {type(e).__name__}: {str(e)[:100]} (fail {fails})")
            if fails > 6:
                if log:
                    log(f"[{src.name}] dropped")
                return
            time.sleep(min(30, 2 ** fails))


# ---- row-type mix: codes share per modality -> sampling probability per row type ----
CODE_SHARE = {"text": 0.40, "image": 0.35, "audio": 0.25}      # of live codes
TEXT_PROMPT_FRAC = 0.15                                          # of text codes that are prompt->response
def type_probs():
    share = {"text0": CODE_SHARE["text"] * (1 - TEXT_PROMPT_FRAC), "text": CODE_SHARE["text"] * TEXT_PROMPT_FRAC,
             "image": CODE_SHARE["image"], "audio": CODE_SHARE["audio"]}
    p = {r: share[r] / LAYOUTS[r].Tt for r in ROW_TYPES}                # codes -> rows
    z = sum(p.values())
    return {r: v / z for r, v in p.items()}
TYPE_P = type_probs()


class MixDataset:
    """Samples sources by (row-type probability x source weight) and yields raw rows for one worker."""

    def __init__(self, spec, kinds=None, part0=1, nparts=2, seed=0, even=False):
        self.spec, self.part0, self.nparts, self.seed = spec, part0, nparts, seed
        srcs = [s for s in SOURCES if kinds is None or s.kind in kinds]
        tw = {}
        for s in srcs:
            tw[s.kind] = tw.get(s.kind, 0) + s.weight
        self.srcs = srcs
        self.p = [(1.0 if even else TYPE_P[s.kind]) * s.weight / tw[s.kind] for s in srcs]   # even: equal share per row type

    def rows(self, wid=0):
        part = self.part0 + wid                      # worker w reads partition part0 + w
        rng = random.Random(self.seed * 7919 + part)
        log = lambda m: print(f"  [stream w{wid}] {m}", file=sys.stderr, flush=True)
        gens = {}
        alive = list(range(len(self.srcs)))
        while alive:
            i = rng.choices(alive, [self.p[j] for j in alive])[0]
            if i not in gens:
                gens[i] = rows_of(self.srcs[i], self.spec, part, self.nparts, self.seed * 31 + part, rng, log)
            try:
                yield next(gens[i])
            except StopIteration:
                alive.remove(i)


def _stream_worker(ds, wid, q, stop):
    torch.set_num_threads(1)                         # a forked child must not inherit the parent's OpenMP pool
    try:
        for row in ds.rows(wid):
            while not stop.is_set():
                try:
                    q.put(row, timeout=1.0)
                    break
                except queue.Full:
                    pass
            if stop.is_set():
                return
    except Exception as e:
        print(f"  [stream w{wid}] worker died: {e!r}", file=sys.stderr, flush=True)


class RowStream:
    """Rows from `workers` forked processes into one queue, delivered in ARRIVAL order.

    (A torch DataLoader hands out worker batches strictly round-robin, so one worker that is busy
    opening a slow dataset stalls every other worker. Streaming needs the opposite.)
    """

    def __init__(self, ds, workers, depth=256):
        self.ds, self.workers = ds, workers
        if workers == 0:
            self.gen = ds.rows(0)
            return
        import multiprocessing as mp
        ctx = mp.get_context("fork")
        self.q, self.stop = ctx.Queue(maxsize=depth), ctx.Event()
        self.procs = [ctx.Process(target=_stream_worker, args=(ds, w, self.q, self.stop), daemon=True)
                      for w in range(workers)]
        for pr in self.procs:
            pr.start()

    def __iter__(self):
        return self

    def __next__(self):
        if self.workers == 0:
            return next(self.gen)
        while True:
            try:
                return self.q.get(timeout=2.0)
            except queue.Empty:
                if not any(pr.is_alive() for pr in self.procs):
                    raise RuntimeError("all stream workers exited")

    def close(self):
        if self.workers:
            self.stop.set()
            time.sleep(0.2)
            for pr in self.procs:
                if pr.is_alive():
                    pr.terminate()


# ---- rows -> GPU codes ----
def micro_batch_sizes():
    return {r: max(1, MICRO_TOKENS // LAYOUTS[r].T) for r in ROW_TYPES}
MICRO_B = micro_batch_sizes()


@torch.no_grad()
def encode_rows(rtype, rows, codecs, device=dev):
    """list of raw rows (one row type) -> dict(rtype, cond [B,Lc] long|None, codes list, n_live)."""
    B = len(rows)
    cond = None
    if LAYOUTS[rtype].cond:
        cond = torch.from_numpy(np.stack([r["cond"] for r in rows])).to(device).long() + 1
    if ROW_MOD[rtype] == "text":
        codes = codecs.text.encode(torch.from_numpy(np.stack([r["tgt"] for r in rows])).to(device))
    elif ROW_MOD[rtype] == "image":
        S = SPEC.side
        imgs = [None] * B
        jidx = [i for i, r in enumerate(rows) if "jpeg" in r]
        if jidx:
            dec = decode_jpeg_gpu([rows[i]["jpeg"] for i in jidx], S, device)
            for n, i in enumerate(jidx):
                imgs[i] = dec[n]
        for i, r in enumerate(rows):
            if imgs[i] is None:
                imgs[i] = torch.from_numpy(r["img"]).to(device)
        codes = codecs.image.encode(torch.stack(imgs))
    else:
        w = torch.from_numpy(np.stack([r["wav"] for r in rows])).to(device).float() / 32768.0
        codes = codecs.audio.encode(w)
    return dict(rtype=rtype, cond=cond, codes=codes, n_live=B * LAYOUTS[rtype].Tt,
                src=[r.get("src", "?") for r in rows])


class Prefetcher:
    """Background thread: loader rows -> per-type micro-batches -> GPU codecs (side stream) -> RAM ring."""

    def __init__(self, loader, codecs, depth=PREFETCH):
        self.q = queue.Queue(maxsize=depth)
        self.loader, self.codecs = loader, codecs   # loader: any iterator of raw rows (RowStream)
        self.stop = threading.Event()
        self.err = None
        self.rows_in = 0
        self.stream = torch.cuda.Stream() if IS_CUDA else None
        self.t = threading.Thread(target=self._run, daemon=True)
        self.t.start()

    def _run(self):
        try:
            bufs = {r: [] for r in ROW_TYPES}
            for row in self.loader:
                if self.stop.is_set():
                    return
                self.rows_in += 1
                b = bufs[row["t"]]
                b.append(row)
                if len(b) >= MICRO_B[row["t"]]:
                    rows, bufs[row["t"]] = b, []
                    if self.stream is not None:
                        with torch.cuda.stream(self.stream):
                            mb = encode_rows(row["t"], rows, self.codecs)
                        self.stream.synchronize()
                    else:
                        mb = encode_rows(row["t"], rows, self.codecs)
                    while not self.stop.is_set():
                        try:
                            self.q.put(mb, timeout=1.0)
                            break
                        except queue.Full:
                            pass
        except Exception as e:                                  # surfaced to the trainer instead of a silent stall
            self.err = e

    def get(self, timeout=600):
        t0 = time.time()
        while True:
            try:
                mb = self.q.get(timeout=1.0)
                break
            except queue.Empty:
                if self.err is not None:
                    raise RuntimeError(f"data thread died: {self.err!r}")
                if time.time() - t0 > timeout:
                    raise TimeoutError(f"no batch for {timeout}s: streams are stalled or throttled (set HF_TOKEN)")
        if IS_CUDA:                                              # tensors were born on the side stream
            cur = torch.cuda.current_stream()
            for t in ([mb["cond"]] if mb["cond"] is not None else []) + mb["codes"]:
                t.record_stream(cur)
        return mb

    def close(self):
        self.stop.set()


def collect_rows(kinds, quota, part0, nparts, workers, timeout=1800, seed=0, even=False):
    """Pull `quota[row_type]` raw rows into RAM from partitions part0.. and stop. Nothing touches disk."""
    ds = MixDataset(SPEC, kinds=kinds, part0=part0, nparts=nparts, seed=seed, even=even)
    out, t0, last = [], time.time(), time.time()
    cnt = {k: 0 for k in quota}
    stream = RowStream(ds, workers)
    try:
        for row in stream:
            t = row["t"]
            if t in cnt and cnt[t] < quota[t]:
                out.append(row); cnt[t] += 1
            if all(cnt[k] >= quota[k] for k in cnt):
                break
            if time.time() - last > 20:
                print(f"    collecting {cnt} / {quota}  ({time.time() - t0:.0f}s)", flush=True); last = time.time()
            if time.time() - t0 > timeout:
                print(f"    !! timeout: got {cnt} of {quota}")
                break
    finally:
        stream.close()
    return out

## 7. Build, size check, codec fit, held-out set

The parameter report comes from a **meta-device** build (no memory), so it prints even on a machine that
cannot hold the model. The real build then checks that fp32 weights + AdamW (16 bytes/param) fit.

In [ ]:
net_meta = build_model(meta=True)
REPORT = param_report(net_meta)
del net_meta
M = 1e6
print(f"total {REPORT['total']/M:,.1f}M | trunk {REPORT['trunk']/M:.1f}M | experts {REPORT['experts_total']/M:,.1f}M "
      f"= 6 x {REPORT['per_expert']/M:.1f}M")
print("active per token:", {m: f"{v/M:.1f}M" for m, v in REPORT["active"].items()},
      f"| capacity {MOE_CF} x 2 experts = {2*MOE_CF:.1f} active expert(s)/token")
if PRESET == "full":
    assert abs(REPORT["total"] - 1.68e9) / 1.68e9 < 0.05, "full preset drifted from the 1.2B+ / ~360M-active target"
    assert 330e6 < REPORT["active_max"] < 390e6, f"active params {REPORT['active_max']/M:.0f}M"
need_gb = REPORT["total"] * 16 / 2 ** 30
print(f"fp32 weights + grads + AdamW ~ {need_gb:.1f} GiB before activations | device has {VRAM_GB:.0f} GiB")
META_ONLY = _knob("META_ONLY", False)
if IS_CUDA and not META_ONLY and VRAM_GB < need_gb * 1.1:
    raise SystemExit(f"PRESET='{PRESET}' needs ~{need_gb:.0f} GiB of VRAM; this GPU has {VRAM_GB:.0f}. "
                     f"Use PRESET='tiny' or an A100/H100 (see the budget cell).")

In [ ]:
if not META_ONLY:
    net = build_model().to(dev)
    net.lay = {r: LAYOUTS[r].to(dev) for r in ROW_TYPES}
    torch._dynamo.config.cache_size_limit = 64

    def _try_flex():
        """Use flex-attention only if it compiles, matches SDPA, and is not slower."""
        from torch.nn.attention.flex_attention import create_block_mask
        for l in net.lay.values():
            bid = l.block_id
            l.flex_mask = create_block_mask(lambda b, h, q, kv, bid=bid: bid[kv] <= bid[q], None, None, l.T, l.T, device=dev)
        blk = net.blocks[0].attn
        r = "audio"
        x = torch.randn(2, net.lay[r].T, P["d"], device=dev, dtype=AMP_DTYPE)
        with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
            a = blk(x, net.lay[r], False).float()
            b = blk(x, net.lay[r], True).float()
        ok = torch.allclose(a, b, atol=5e-2, rtol=5e-2)
        for rr in ROW_TYPES:                                   # every packed geometry, forward AND backward: each compiles its own kernel
            xg = torch.randn(2, net.lay[rr].T, P["d"], device=dev, dtype=AMP_DTYPE, requires_grad=True)
            with torch.enable_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
                blk(xg, net.lay[rr], True).float().sum().backward()
        net.zero_grad(set_to_none=True)
        def bench(use):
            for _ in range(3):
                blk(x, net.lay[r], use)
            torch.cuda.synchronize(); t = time.time()
            for _ in range(10):
                blk(x, net.lay[r], use)
            torch.cuda.synchronize()
            return (time.time() - t) / 10
        with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
            ts, tf = bench(False), bench(True)
        print(f"attention: sdpa {ts*1e3:.2f} ms | flex {tf*1e3:.2f} ms | parity {ok}")
        return ok and (ATTN == "flex" or tf <= ts * 1.05)

    net.use_flex = False
    _smem_kb = torch.cuda.get_device_properties(0).shared_memory_per_block_optin // 1024 if IS_CUDA else 0
    if IS_CUDA and HAS_FLEX and ATTN == "auto" and _smem_kb < 160:
        print(f"flex_attention skipped: {_smem_kb} KB shared memory per block (A100-class has 163); using sdpa. ATTN=flex forces it.")
    elif IS_CUDA and HAS_FLEX and ATTN != "sdpa":
        try:
            net.use_flex = _try_flex()
        except Exception as e:
            print("flex_attention unavailable:", type(e).__name__, str(e)[:80])
    print("attention backend:", "flex (block mask)" if net.use_flex else "sdpa (bool mask)")

    COMPILED = bool(COMPILE and IS_CUDA)
    loss_fn = torch.compile(net.loss, dynamic=False, mode=COMPILE_MODE) if COMPILED else net.loss
    print(f"model on {dev} | {sum(p.numel() for p in net.parameters())/M:,.1f}M params | torch.compile {COMPILED}")

### Codec fit and held-out set (streamed, held in RAM only)

In [ ]:
if not META_ONLY:
    PARTS = max(WORKERS, 1) + 1                 # part 0 = validation; parts 1.. = training workers
    codecs = Codecs(SPEC, dev)

    def fit_codecs():
        t0 = time.time()
        print(f"fitting frozen codebooks from {FIT_IMAGES} streamed images and {FIT_AUDIO} audio clips (RAM only)...")
        rows = collect_rows(("image",), {"image": FIT_IMAGES}, 1, PARTS, WORKERS)
        feats, B = [], 64
        for i in range(0, len(rows), B):
            sub = [r for r in rows[i:i + B]]
            imgs = []
            jp = [j for j, r in enumerate(sub) if "jpeg" in r]
            tmp = [None] * len(sub)
            if jp:
                d = decode_jpeg_gpu([sub[j]["jpeg"] for j in jp], SPEC.side, dev)
                for n, j in enumerate(jp):
                    tmp[j] = d[n]
            for j, r in enumerate(sub):
                if tmp[j] is None:
                    tmp[j] = torch.from_numpy(r["img"]).to(dev)
            feats.append(sample_points(codecs.image.features(torch.stack(tmp)), 200_000 // max(1, len(rows) // B + 1)))
        codecs.image.fit(feats); del rows, feats
        print(f"  image codebooks done ({time.time() - t0:.0f}s)")
        rows = collect_rows(("audio",), {"audio": FIT_AUDIO}, 1, PARTS, WORKERS)
        waves = [torch.from_numpy(np.stack([r["wav"] for r in rows[i:i + 32]])).to(dev).float() / 32768 for i in range(0, len(rows), 32)]
        codecs.audio.set_stats(waves)
        codecs.audio.fit([sample_points(codecs.audio.features(w), 200_000 // max(1, len(waves))) for w in waves])
        print(f"  audio codebooks done ({time.time() - t0:.0f}s)  mel mean {codecs.audio.mean:.2f} std {codecs.audio.std:.2f}")

    CKPT_PATH = os.path.join(CKPT_DIR, "ckpt.pt") if CKPT_DIR else ""
    RESUME = None
    if CKPT_PATH and os.path.exists(CKPT_PATH):
        RESUME = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
        codecs.load(RESUME["codecs"])
        print(f"resuming from {CKPT_PATH}: step {RESUME['step']}, codebooks restored (no refit)")
    else:
        fit_codecs()

    def build_val():
        t0 = time.time()
        quota = {r: VAL_ROWS for r in ROW_TYPES}
        rows = collect_rows(None, quota, 0, PARTS, 1, seed=12345, even=True)
        out = []
        for r in ROW_TYPES:
            rr = [x for x in rows if x["t"] == r]
            for i in range(0, len(rr), min(MICRO_B[r], 16)):
                out.append(encode_rows(r, rr[i:i + min(MICRO_B[r], 16)], codecs))
        print(f"held-out set: { {r: sum(len(b['src']) for b in out if b['rtype'] == r) for r in ROW_TYPES} } rows ({time.time() - t0:.0f}s)")
        return out

    VAL = build_val()

## 8. Budget: tiers, steps, throughput, ETA (printed before anything expensive runs)

Compute-optimal is ~20 tokens per active parameter (~7B here); capable small models are trained 10-100x
past that. A "live code" is one predicted (non-prefix) position.

| tier | live codes | text | image | audio | tokens / active param | what it buys |
|---|---|---|---|---|---|---|
| T0 smoke | 0.02B | 40% | 35% | 25% | 0.06 | pipeline works, loss moves |
| T1 pilot | 2B | | | | 5.5 | per-modality CE separates, coarse levels sensible |
| **T2 foundation-lite (default)** | **20B** | 8B | 7B | 5B | 55 | readable text, recognisable image structure, intelligible TTS |
| T3 foundation | 50B | 20B | 18B | 12.5B | 140 | still far below the 1-4T tokens published ~360M text LMs saw |

Wall-clock is `6 * N_active * D / sustained FLOPS`: ~3.6 days on an A100 at 45% MFU, ~1.3 days on an
H100 for T2. The measured numbers below replace these estimates.

In [ ]:
TIERS = {"T0": 0.02, "T1": 2.0, "T2": 20.0, "T3": 50.0}
TOK_B = TOKENS or TIERS[TIER]
AVG_LIVE_MICRO = sum(TYPE_P[r] * MICRO_B[r] * LAYOUTS[r].Tt for r in ROW_TYPES)
ACCUM = max(1, round(BATCH_CODES / AVG_LIVE_MICRO))
CODES_PER_STEP = ACCUM * AVG_LIVE_MICRO
STEPS_TOTAL = STEPS or max(1, round(TOK_B * 1e9 / CODES_PER_STEP))
TOK_B = STEPS_TOTAL * CODES_PER_STEP / 1e9

rows_per_code = {r: 1.0 / LAYOUTS[r].Tt for r in ROW_TYPES}
tgt = {"text": TOK_B * CODE_SHARE["text"], "image": TOK_B * CODE_SHARE["image"], "audio": TOK_B * CODE_SHARE["audio"]}
n_text = tgt["text"] * 1e9 / SPEC.text / (1)                       # approx rows of text (live codes per row = ladder total)
print(f"tier {TIER}: {TOK_B:.3f}B live codes = {STEPS_TOTAL:,} optimizer steps x {CODES_PER_STEP:,.0f} codes "
      f"({ACCUM} micro-batches of ~{MICRO_TOKENS} packed positions)")
print(f"tokens per active parameter: {TOK_B*1e9/REPORT['active_max']:.1f} (compute-optimal ~20)")
for m in ("text", "image", "audio"):
    live = LAYOUTS["text0" if m == "text" else m].Tt
    rows = tgt[m] * 1e9 / live
    raw = {"text": f"~{rows * SPEC.text / 1e9:.1f} GB of text", "image": f"~{rows/1e6:.2f}M images",
           "audio": f"<= {rows * SPEC.wave_len / SPEC.sr / 3600:,.0f} h of audio (incl. silence padding)"}[m]
    print(f"  {m:5s} {tgt[m]:7.3f}B live codes = {rows/1e6:8.2f}M rows  ({raw})")
print("row-type mix (rows):", {r: f"{TYPE_P[r]:.3f}" for r in ROW_TYPES})

FLOPS_TOTAL = 6 * REPORT["active_max"] * TOK_B * 1e9 * (sum(TYPE_P[r] * LAYOUTS[r].T for r in ROW_TYPES)
                                                         / sum(TYPE_P[r] * LAYOUTS[r].Tt for r in ROW_TYPES))
print(f"training FLOPs (6 N_active x packed tokens): {FLOPS_TOTAL:.2e}")
for nm, tf in (("A100 @45% MFU", 140), ("H100 bf16 @40%", 400), ("RTX 3070 (measured below)", None)):
    if tf:
        print(f"  {nm:26s} ~{FLOPS_TOTAL / (tf * 1e12) / 3600:7.1f} h")

In [ ]:
def synth(rtype, B):
    lay = LAYOUTS[rtype]
    codes = [torch.randint(1, 257, (B, n), device=dev) for n in lay.lens]
    cond = torch.randint(1, 257, (B, lay.cond), device=dev) if lay.cond else None
    return cond, codes


def _amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if (IS_CUDA and BF16) else torch.autocast("cpu", enabled=False)


def bench_compute(iters=3):
    """Measured fwd+bwd time per micro-batch of each row type -> blended live codes/s, packed tokens/s, MFU."""
    net.train()
    t_of = {}
    for r in ROW_TYPES:
        B = MICRO_B[r]
        cond, codes = synth(r, B)
        for i in range(iters + 2):
            if i == 2:
                if IS_CUDA: torch.cuda.synchronize()
                t0 = time.time()
            with _amp():
                loss, _ = loss_fn(r, cond, codes)
            loss.backward()
        if IS_CUDA: torch.cuda.synchronize()
        t_of[r] = (time.time() - t0) / iters
        net.zero_grad(set_to_none=True)
    t_avg = sum(TYPE_P[r] * t_of[r] for r in ROW_TYPES)
    live_s = AVG_LIVE_MICRO / t_avg
    tok_s = sum(TYPE_P[r] * MICRO_B[r] * LAYOUTS[r].T for r in ROW_TYPES) / t_avg
    tflops = 6 * REPORT["active_max"] * tok_s / 1e12
    return t_of, live_s, tok_s, tflops


if not META_ONLY:
    t_of, LIVE_PER_S, TOK_PER_S, TFLOPS = bench_compute()
    mfu = f" | MFU {100 * TFLOPS / PEAK_TFLOPS:.0f}% of ~{PEAK_TFLOPS} peak" if PEAK_TFLOPS else ""
    print("seconds per micro-batch:", {r: round(t, 3) for r, t in t_of.items()})
    print(f"compute: {LIVE_PER_S:,.0f} live codes/s | {TOK_PER_S:,.0f} packed tokens/s | {TFLOPS:.1f} TFLOPS{mfu}")
    if IS_CUDA:
        print(f"peak memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB of {VRAM_GB:.0f}")
    eta_h = TOK_B * 1e9 / LIVE_PER_S / 3600
    need_rows = LIVE_PER_S / (sum(TYPE_P[r] * LAYOUTS[r].Tt for r in ROW_TYPES))
    print(f"==> {TOK_B:.3f}B codes would take ~{eta_h:,.1f} h on this device if the feed keeps up; "
          f"the feed must deliver ~{need_rows:,.0f} rows/s ({WORKERS} workers -> ~{need_rows / max(WORKERS, 1):.1f} rows/s each)")

## 9. Correctness: fails loudly and cheaply, before any streamed token is spent

1. **No level leakage.** Perturbing the codes of level *j* leaves the hidden state of every slot at
   levels <= *j* unchanged and moves level *j+1* (so the model cannot copy its own targets).
2. **The prefix is a real condition.** Perturbing the prompt/caption/transcript moves level 0, and
   perturbing targets never moves the prefix.
3. **Modality isolation.** A text-only batch sends exactly zero gradient into the image and audio experts.
4. **Expert capacity.** Coverage stats: every token gets 0, 1 or 2 experts, about 1 on average.
5. **Kernel parity.** Triton SwiGLU vs reference, chunked CE vs `F.cross_entropy`, flex vs SDPA (printed above).
6. **Codec round trip** and an initial loss at the uniform-prediction value `ln(257)`.

In [ ]:
def _fresh_codes(rtype, B, seed):
    g = torch.Generator(device=dev).manual_seed(seed)
    lay = LAYOUTS[rtype]
    codes = [torch.randint(1, 257, (B, n), device=dev, generator=g) for n in lay.lens]
    cond = torch.randint(1, 257, (B, lay.cond), device=dev, generator=g) if lay.cond else None
    return cond, codes


@torch.no_grad()
def check_no_leak(net, rtype):
    lay = LAYOUTS[rtype]
    cond, codes = _fresh_codes(rtype, 2, 1)
    h0 = net.hidden(rtype, cond, codes)
    worst = {}
    for j in range(lay.K):
        codes2 = [c.clone() for c in codes]
        codes2[j] = (codes2[j] % 256) + 1                                  # every code at level j changes
        h1 = net.hidden(rtype, cond, codes2)
        upto = lay.lvl[j][1]
        same = (h0[:, :upto] - h1[:, :upto]).abs().max().item()            # prefix + levels <= j
        assert same < 1e-5, f"{rtype}: perturbing level {j} moved a slot at level <= {j} by {same:.2e} (LEAK)"
        if j + 1 < lay.K:
            a, b = lay.lvl[j + 1]
            moved = (h0[:, a:b] - h1[:, a:b]).abs().max().item()
            assert moved > 1e-4, f"{rtype}: level {j+1} ignores level {j} ({moved:.2e})"
            worst[j] = round(moved, 3)
    if lay.cond:
        cond2 = (cond % 256) + 1
        h1 = net.hidden(rtype, cond2, codes)
        a, b = lay.lvl[0]
        moved = (h0[:, a:b] - h1[:, a:b]).abs().max().item()
        assert moved > 1e-4, f"{rtype}: level 0 ignores the prefix"
    return worst


if not META_ONLY:
    net.eval()
    for r in ROW_TYPES:
        w = check_no_leak(net, r)
        print(f"  no-leak ok: {r:6s} level->level influence { w }")

    # (6) initial loss: heads start at zero, so the first prediction is uniform and the loss is ln(257)
    net.eval()
    with torch.no_grad():
        l0, _ = net.loss("text0", *_fresh_codes("text0", 2, 2))
    assert abs(l0.item() - math.log(257)) < 0.05, f"init loss {l0.item():.3f} != ln(257)={math.log(257):.3f}"
    print(f"  init loss {l0.item():.3f} ~ ln(257) = {math.log(257):.3f}")

    # (3) modality isolation. A zero head sends exactly zero gradient into the trunk on step 0, so the test
    # swaps in a small random head, checks the gradient, and puts the zero head back.
    saved = {k: v.clone() for k, v in net.heads.state_dict().items()}
    for hd in net.heads.values():
        nn.init.normal_(hd.weight, std=0.02)
    net.train(); net.zero_grad(set_to_none=True)
    cond, codes = _fresh_codes("text0", 2, 2)
    loss, _ = net.loss("text0", cond, codes)
    loss.backward()
    g = {m: sum(float(p.grad.abs().sum()) if p.grad is not None else 0.0
                for blk in net.blocks for p in blk.experts[m].parameters()) for m in ("text", "image", "audio")}
    assert g["text"] > 0 and g["image"] == 0 and g["audio"] == 0, f"modality isolation broken: {g}"
    print("  isolation ok: text0 batch ->", {m: ("grad" if v > 0 else "zero grad") for m, v in g.items()})
    net.zero_grad(set_to_none=True)
    net.heads.load_state_dict(saved)

    # (4) coverage
    net.eval()
    for blk in net.blocks:
        for e in blk.experts.values():
            e.record = True
    with torch.no_grad():
        net.hidden("image", *_fresh_codes("image", 4, 3)[:1], _fresh_codes("image", 4, 3)[1])
    cov = {m: net.blocks[-1].experts[m].last_cov for m in ("text", "image")}
    for blk in net.blocks:
        for e in blk.experts.values():
            e.record = False
    for m, c in cov.items():
        if c is not None:
            mean = float(c[1] + 2 * c[2])
            print(f"  coverage (last block, {m} group): 0 experts {c[0]:.2f} | 1 expert {c[1]:.2f} | 2 experts {c[2]:.2f} | mean {mean:.2f}")
            assert 0.9 < mean < 1.3

    # (5) kernels
    h = torch.randn(64, 2 * 48, device=dev, dtype=torch.float32, requires_grad=True)
    ref = swiglu_torch(h)
    assert torch.allclose(swiglu(h), ref, atol=1e-4, rtol=1e-4)
    W, b_ = torch.randn(257, 32, device=dev), torch.randn(257, device=dev)
    hh, tt = torch.randn(1000, 32, device=dev), torch.randint(1, 257, (1000,), device=dev)
    _tf32 = torch.backends.cuda.matmul.allow_tf32      # TF32 rounds the two matmuls differently (~1e-2), which hides real bugs
    torch.backends.cuda.matmul.allow_tf32 = False
    try:
        ce = fused_ce(hh, W, b_, tt, chunk=300)[0]
        ref_ce = F.cross_entropy(F.linear(hh.double(), W.double(), b_.double()), tt, reduction="none")
        err = (ce.double() - ref_ce).abs().max().item()
        assert err < 1e-3, f"chunked CE differs from reference by {err:.2e}"
    finally:
        torch.backends.cuda.matmul.allow_tf32 = _tf32
    print(f"  kernels ok: swiglu[{SWIGLU_IMPL}] == reference, chunked CE == F.cross_entropy")

    # (6) codecs
    txt = torch.randint(0, 256, (2, SPEC.text), dtype=torch.uint8, device=dev)
    assert TextCodec.decode(codecs.text.encode(txt)[-1])[0] == bytes(txt[0].tolist()).rstrip(b"\x00").decode("utf-8", "replace")
    print("  codec round trip ok (text exact); sizes:", {r: LAYOUTS[r].Tt for r in ROW_TYPES})
    print("\nALL CORRECTNESS CHECKS PASSED")
    net.train()

## 10. Train

* One optimizer step = `ACCUM` micro-batches (each a single row type, a fixed token budget) -> ~`BATCH_CODES` live codes.
* AdamW (fused), beta (0.9, 0.95), wd on matrices only, linear warmup then cosine to 10%, grad-clip, bf16 autocast.
* Logged every `LOG_EVERY` steps: loss, per-(modality, level) CE, tokens/s, TFLOPS/MFU, and the **share of the step spent
  waiting for data**, which tells a feed-bound run from a compute-bound one.
* Held-out evaluation every `EVAL_EVERY_B` billion codes against the **constant-predictor baseline**: the cross-entropy of the
  held-out codes under the *training* unigram of that (modality, level). A score under it is not a result.
* Checkpoints (weights, optimizer, step, codes seen, codebooks) go to `CKPT_DIR` every `CKPT_MIN` minutes and a watchdog saves
  and stops before `SESSION_HOURS`. Data is never saved; a resume re-streams with a fresh shuffle seed ("approximately resume").
* Early abort: if the loss has not fallen 5% by step 200, stop instead of burning the budget.

In [ ]:
HIST = []
UNI = {}          # (modality, level) -> running histogram of TRAINING target codes, the honest constant predictor


def make_optimizer():
    dec, nodec = [], []
    for n, p_ in net.named_parameters():
        (dec if p_.ndim >= 2 and not n.startswith(("emb", "pos")) and "router" not in n else nodec).append(p_)
    return torch.optim.AdamW([dict(params=dec, weight_decay=WD), dict(params=nodec, weight_decay=0.0)],
                             lr=LR, betas=(0.9, 0.95), eps=1e-8, fused=IS_CUDA)


WARM = min(WARMUP, max(1, STEPS_TOTAL // 5))
def lr_at(step):
    if step < WARM:
        return LR * (step + 1) / WARM
    prog = min(1.0, (step - WARM) / max(1, STEPS_TOTAL - WARM))
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))


@torch.no_grad()
def evaluate():
    """Teacher-forced held-out CE / top-1 per (modality, level) + the constant-predictor baseline."""
    net.eval()
    acc = {}
    for mb in VAL:
        r = mb["rtype"]; m = ROW_MOD[r]
        with _amp():
            h = net.hidden(r, mb["cond"], mb["codes"])
        for k in range(LAYOUTS[r].K):
            lg = net.level_logits(r, h, k)
            tg = mb["codes"][k]
            ce = F.cross_entropy(lg.reshape(-1, SPEC.V), tg.reshape(-1), reduction="none")
            a = acc.setdefault((m, k), dict(ce=0.0, ok=0.0, n=0, cnt=torch.zeros(SPEC.V, device=dev)))
            a["ce"] += float(ce.sum()); a["ok"] += float((lg.argmax(-1) == tg).sum()); a["n"] += tg.numel()
            a["cnt"] += torch.bincount(tg.reshape(-1), minlength=SPEC.V).float()
    out = {}
    for (m, k), a in acc.items():
        fv = a["cnt"] / a["cnt"].sum()                              # held-out code frequencies
        u = UNI.get((m, k))
        if u is not None and float(u.sum()) > 0:                    # constant predictor = training unigram, add-one smoothed
            lp = ((u + 1) / (u.sum() + SPEC.V)).log()
            base = float(-(fv * lp).sum())
        else:                                                       # nothing trained yet: held-out entropy (optimistic)
            base = float(-(fv[fv > 0] * fv[fv > 0].log()).sum())
        out[f"{m}_L{k}"] = dict(ce=a["ce"] / a["n"], acc=a["ok"] / a["n"], base=base, n=a["n"])
    net.train()
    return out


def save_ckpt(step, opt, seen):
    if not CKPT_DIR:
        return
    os.makedirs(CKPT_DIR, exist_ok=True)
    tmp = CKPT_PATH + ".tmp"
    torch.save(dict(net=net.state_dict(), opt=opt.state_dict(), step=step, seen=seen, hist=HIST[-2000:],
                    codecs=codecs.state(), preset=PRESET), tmp)
    os.replace(tmp, CKPT_PATH)
    print(f"  checkpoint -> {CKPT_PATH} (step {step})", flush=True)


def fmt_val(v):
    mods = {}
    for key, d in v.items():
        mods.setdefault(key.split("_")[0], []).append(d)
    return " | ".join(f"{m} ce {np.mean([d['ce'] for d in ds]):.2f} (base {np.mean([d['base'] for d in ds]):.2f}, "
                      f"acc {100*np.mean([d['acc'] for d in ds]):.0f}%)" for m, ds in mods.items())


def train():
    global HIST
    net.train()
    opt = make_optimizer()
    seen, step0 = dict(text=0, image=0, audio=0), 0
    if RESUME is not None:
        net.load_state_dict(RESUME["net"]); opt.load_state_dict(RESUME["opt"])
        step0, seen, HIST = RESUME["step"], RESUME["seen"], RESUME["hist"]
        print(f"resumed at step {step0}, codes seen {seen}")
    eval_every = max(1, min(round(EVAL_EVERY_B * 1e9 / CODES_PER_STEP), max(1, STEPS_TOTAL // 3)))
    ds = MixDataset(SPEC, part0=1, nparts=PARTS, seed=SEED + 1 + step0)
    stream = RowStream(ds, WORKERS)
    pf = Prefetcher(stream, codecs)
    print(f"streaming with {WORKERS} workers; first batch can take a minute while sources connect...", flush=True)
    t_run, t_ckpt, t_log, wait, steps_in_log = time.time(), time.time(), time.time(), 0.0, 0
    lvl_sum, lvl_n, loss_sum = {}, {}, torch.zeros((), device=dev)
    live_in_log, packed_in_log = 0, 0
    abort_at = 200 if STEPS_TOTAL >= 400 else None
    try:
        for step in range(step0, STEPS_TOTAL):
            for g in opt.param_groups:
                g["lr"] = lr_at(step)
            opt.zero_grad(set_to_none=True)
            for _ in range(ACCUM):
                t0 = time.time()
                mb = pf.get()
                wait += time.time() - t0
                with _amp():
                    loss, per_level = loss_fn(mb["rtype"], mb["cond"], mb["codes"])
                (loss / ACCUM).backward()
                m = ROW_MOD[mb["rtype"]]
                loss_sum += loss.detach() / ACCUM
                lvl_sum[(m, mb["rtype"])] = lvl_sum.get((m, mb["rtype"]), 0) + per_level
                lvl_n[(m, mb["rtype"])] = lvl_n.get((m, mb["rtype"]), 0) + 1
                seen[m] += mb["n_live"]; live_in_log += mb["n_live"]
                for k_, c_ in enumerate(mb["codes"]):
                    UNI[(m, k_)] = UNI.get((m, k_), 0) + torch.bincount(c_.reshape(-1), minlength=SPEC.V).float()
                packed_in_log += len(mb["src"]) * LAYOUTS[mb["rtype"]].T
            gn = torch.nn.utils.clip_grad_norm_(net.parameters(), CLIP, foreach=True)
            opt.step()
            steps_in_log += 1
            done = step + 1
            if done % LOG_EVERY == 0 or done == STEPS_TOTAL or done == 1:
                if IS_CUDA: torch.cuda.synchronize()
                dt = time.time() - t_log
                loss_v = float(loss_sum) / steps_in_log
                ce = {}
                for (m, r), v in lvl_sum.items():
                    for k, x in enumerate((v / lvl_n[(m, r)]).tolist()):
                        ce.setdefault(f"{m}_L{k}", []).append(x)
                ce = {k: round(float(np.mean(v)), 4) for k, v in ce.items()}
                tok_s = packed_in_log / dt
                tfl = 6 * REPORT["active_max"] * tok_s / 1e12
                rec = dict(step=done, loss=loss_v, lr=lr_at(step), gnorm=float(gn), codes_s=live_in_log / dt,
                           tflops=tfl, wait=wait / dt, seen=dict(seen), ce=ce)
                HIST.append(rec)
                eta = (STEPS_TOTAL - done) * dt / steps_in_log / 3600
                mfu = f" ({100*tfl/PEAK_TFLOPS:.0f}% MFU)" if PEAK_TFLOPS else ""
                print(f"step {done}/{STEPS_TOTAL} | loss {loss_v:.3f} | lr {rec['lr']:.1e} | gn {rec['gnorm']:.2f} | "
                      f"{rec['codes_s']/1e3:.1f}k codes/s | {tfl:.1f} TFLOPS{mfu} | data-wait {100*rec['wait']:.0f}% | "
                      f"seen {sum(seen.values())/1e9:.4f}B | eta {eta:.1f}h", flush=True)
                if rec["wait"] > 0.5 and done > 3 and (done // LOG_EVERY) % 10 == 1:
                    print(f"  !! feed-bound: the GPU waits {100*rec['wait']:.0f}% of each step for data. "
                          f"Raise WORKERS (now {WORKERS}) or run where more CPU cores / bandwidth are available.", flush=True)
                lvl_sum, lvl_n, loss_sum = {}, {}, torch.zeros((), device=dev)
                live_in_log = packed_in_log = steps_in_log = 0
                wait, t_log = 0.0, time.time()
                if not math.isfinite(loss_v):
                    raise FloatingPointError("non-finite loss: stop and lower LR")
            if abort_at and done == abort_at:
                tr = [h["loss"] for h in HIST if "loss" in h and h["step"] <= done]   # val records have no "loss"
                if len(tr) >= 8 and np.mean(tr[-3:]) > 0.95 * np.mean(tr[:3]):
                    print("NOT LEARNING: loss fell <5% in 200 steps. Stop and check the CE table before spending the budget.")
                    print(HIST[-1]["ce"])
                    return False
            if done % eval_every == 0 or done == STEPS_TOTAL:
                v = evaluate()
                HIST.append(dict(step=done, val=v))
                print(f"  eval @ {done}: {fmt_val(v)}", flush=True)
            if CKPT_DIR and time.time() - t_ckpt > CKPT_MIN * 60:
                save_ckpt(done, opt, seen); t_ckpt = time.time()
            if time.time() - t_run > SESSION_HOURS * 3600:
                save_ckpt(done, opt, seen)
                print(f"watchdog: {SESSION_HOURS} h reached at step {done}. Checkpoint saved: re-run this notebook with the same "
                      f"CKPT_DIR to resume.")
                return True
        save_ckpt(STEPS_TOTAL, opt, seen)
    finally:
        pf.close(); stream.close()
    return True


if not META_ONLY:
    TRAINED = train()

## 11. Did it learn? (verdict from the run's own records)

In [ ]:
def verdict():
    tr = [h for h in HIST if "loss" in h]
    va = [h for h in HIST if "val" in h]
    fails = []
    def need(ok, why):
        if not ok: fails.append(why)
    need(len(tr) >= 1, "no logged steps")
    need(all(math.isfinite(h["loss"]) and math.isfinite(h["gnorm"]) for h in tr), "non-finite loss or grad norm")
    short = len(tr) < 6
    if not short:
        n = max(1, len(tr) // 5)
        early, late = np.median([h["loss"] for h in tr[:n]]), np.median([h["loss"] for h in tr[-n:]])
        drop = 100 * (1 - late / early)
        print(f"loss  median {early:.3f} -> {late:.3f} ({drop:+.1f}%; ln(257) at init = {math.log(257):.3f})")
        need(late <= 0.83 * early, f"loss moved only {drop:.1f}%, want >= 17%")
        per = {}
        for h in tr:
            for k, v in h["ce"].items():
                per.setdefault(k.split("_")[0], {}).setdefault(k, []).append(v)
        moved = 0
        for m, ks in per.items():
            first = np.mean([v[0] for v in ks.values()]); last = np.mean([v[-1] for v in ks.values()])
            print(f"  {m:5s} train CE {first:.3f} -> {last:.3f} ({100*(1-last/first):+.1f}%)")
            moved += (last <= 0.9 * first)
        need(moved >= 2, f"only {moved}/3 modalities' CE fell >= 10%")
    if va:
        v = va[-1]["val"]
        beat = {}
        for key, d in v.items():
            beat.setdefault(key.split("_")[0], []).append(d["base"] - d["ce"])
        gain = {m: float(np.mean(x)) for m, x in beat.items()}
        print("held-out gain over the constant-predictor baseline (nats, mean over levels):", {m: round(g, 3) for m, g in gain.items()})
        if not short:
            need(sum(g > 0.05 for g in gain.values()) >= 2, f"held-out CE does not beat the constant baseline for 2+ modalities: {gain}")
    print()
    if short:
        print(f"VERDICT: TOO SHORT to judge ({len(tr)} logged steps) | pipeline ran end to end, loss finite" if not fails else f"VERDICT: BROKEN {fails}")
    elif fails:
        print("VERDICT:", "NOT LEARNING" if late > 0.95 * early else "PARTIAL")
        for f in fails: print("  -", f)
    else:
        print(f"VERDICT: PASS | loss {drop:.1f}% down | seen {sum(tr[-1]['seen'].values())/1e9:.3f}B live codes")
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
        ax[0].plot([h["step"] for h in tr], [h["loss"] for h in tr]); ax[0].set_title("train loss"); ax[0].set_xlabel("step")
        for m in ("text", "image", "audio"):
            ys = [np.mean([v for k, v in h["ce"].items() if k.startswith(m)]) for h in tr]
            ax[1].plot([h["step"] for h in tr], ys, label=m)
        ax[1].legend(); ax[1].set_title("train CE per modality"); ax[1].set_xlabel("step"); plt.tight_layout(); plt.show()
    except Exception:
        pass

if not META_ONLY:
    verdict()

## 12. Generate: K forwards per sample, coarse to fine

Level 0 is sampled first (from NULL input, given only the prefix), then each next level is sampled in
one forward from the previous level's sampled codes. The prefix makes the same weights do
**text -> text** (prompt), **text -> image** (caption) and **text -> speech** (transcript = basic TTS).
The vocoder is Griffin-Lim on the decoded log-mel: intelligible at best, robotic by construction.
A short run will produce babble; what these cells prove is that the whole conditional path works.

In [ ]:
@torch.no_grad()
def generate(rtype, prompt=None, n=1, temperature=0.9, top_k=64, seed=0):
    net.eval()
    lay = LAYOUTS[rtype]
    g = torch.Generator(device=dev).manual_seed(seed)
    cond = None
    if lay.cond:
        arr, _ = _bytes(prompt or "", SPEC.cond)
        cond = torch.from_numpy(arr)[None].repeat(n, 1).to(dev).long() + 1
    codes = [torch.zeros(n, l, dtype=torch.long, device=dev) for l in lay.lens]
    for k in range(lay.K):
        with _amp():
            h = net.hidden(rtype, cond, codes)
        lg = net.level_logits(rtype, h, k) / temperature
        lg[..., 0] = float("-inf")                                    # NULL is never a sample
        if top_k:
            kth = lg.topk(top_k, dim=-1).values[..., -1:]
            lg = lg.masked_fill(lg < kth, float("-inf"))
        codes[k] = torch.multinomial(lg.softmax(-1).reshape(-1, SPEC.V), 1, generator=g).view(n, -1)
    net.train()
    return codes


try:
    from IPython.display import display, Audio
except Exception:
    display = print; Audio = None


def show_text(prompt, **kw):
    out = TextCodec.decode(generate("text", prompt, **kw)[-1])[0]
    print(f"[text->text]  {prompt!r}\n  -> {out!r}")


def show_image(caption, n=4, **kw):
    from PIL import Image as PILImage
    codes = generate("image", caption, n=n, **kw)
    S = SPEC.side
    rows = []
    for k in range(SPEC.K):                                           # the coarse-to-fine progression, one row per level
        im = codecs.image.decode(codes[k], level=k).cpu().numpy()
        rows.append(np.concatenate([x.transpose(1, 2, 0) for x in im], 1))
    display(PILImage.fromarray(np.concatenate(rows, 0)))
    print(f"[text->image] {caption!r}: rows = level 0..{SPEC.K-1} (blurry -> detailed), {n} samples")


def tts(text, **kw):
    codes = generate("audio", text, n=1, **kw)
    wav = codecs.audio.decode(codes[-1], gl_iters=60)[0].float().cpu().numpy()
    wav = wav / max(1e-6, abs(wav).max()) * 0.9
    print(f"[text->speech] {text!r}: {len(wav)/SPEC.sr:.1f} s, Griffin-Lim vocoder")
    if Audio is not None:
        display(Audio(wav, rate=SPEC.sr))
    return wav


if not META_ONLY:
    gen_kw = dict(temperature=0.9, top_k=64)
    show_text("what is the capital of france?", **gen_kw)
    show_text("1 + 1 =", **gen_kw)
    show_image("a photo of a dog on the grass", **gen_kw)
    show_image("a scanned document with a table", **gen_kw)
    _ = tts("the quick brown fox jumps over the lazy dog", **gen_kw)

## 13. Nothing was stored

The HF cache lives in RAM and the only files this run may have written are checkpoints (and only if
`CKPT_DIR` is set). This cell lists any new file larger than 100 KB on disk and checks that the in-RAM Hub cache holds only metadata (< 50 MB).

In [ ]:
def _new_files():
    out = []
    roots = [os.getcwd()]                      # disk only: the RAM cache (/dev/shm) is checked by size below
    for root in roots:
        for dp, dn, fn in os.walk(root):
            dn[:] = [d for d in dn if d not in (".git", "__pycache__", ".ipynb_checkpoints", ".venv")]
            if CKPT_DIR and os.path.abspath(dp).startswith(os.path.abspath(CKPT_DIR)):
                continue
            for f in fn:
                p_ = os.path.join(dp, f)
                try:
                    if os.path.getmtime(p_) >= T_START and os.path.getsize(p_) > 100_000 and not f.endswith((".ipynb", ".py", ".md")):
                        out.append((os.path.getsize(p_), p_))
                except OSError:
                    pass
    return sorted(out, reverse=True)

_nf = _new_files()
ram_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(_RAM_HF) for f in fs) / 2 ** 20 if _RAM_HF else 0
print(f"HF cache in RAM (Hub metadata only): {ram_mb:.1f} MB | new files > 100 KB on disk: {len(_nf)}")
for sz, p_ in _nf[:10]:
    print(f"  {sz/2**20:8.2f} MB  {p_}")
assert not _nf, "this run wrote data files to disk; streaming must not store anything"
assert ram_mb < 50, f"HF cache in RAM is {ram_mb:.0f} MB: more than metadata was cached"
print("OK: all data streamed, nothing cached or stored")